# Can any scorer separate variants that improved binding from variants that worsened it?

Every calibration in this repo so far has asked a scorer to **rank**: Spearman
against ddG in `03_adapt_calibration.ipynb`, Spearman against KD within a fixed
substitution count in `03_chinery_size_calibration.ipynb`. Ranking is the harder
question, both notebooks answer it badly, and that leaves the easier question
unanswered and worth asking on its own:

**given one variant measured to bind better and one measured to bind worse, does
the scorer put them in the right order?**

That is a question about direction, not magnitude, and it has one natural
statistic. AUROC *is* the probability that a randomly drawn positive outranks a
randomly drawn negative, so it is the question written as a number rather than a
proxy for it. A scorer at 0.5 is a coin. A scorer that cannot clear 0.5 on
direction has no business filtering a shortlist, whatever its correlation looks
like.

The reason this had not been asked is that the evidence set had almost no
negatives. `03_adapt_calibration.ipynb` leans on two - `L:S50F` and `L:S50W` -
and two negatives among eleven singles put a floor of **p = 1/C(11,2) = 0.018**
under any permutation test, which Part 2 reports so that what the extra
negatives bought is a number and not a feeling.
`data/benchmarks/trastuzumab_negatives.csv` now carries eight measured affinity
losses from the literature and several hundred labelled Chinery wells, which is
what makes this notebook possible.

| Part | What it does | Reads | Writes |
|---|---|---|---|
| 0 | Preflight: which scorers have a route on this box | `routes.py` | nothing |
| 1 | The labelled direction set: 21 positives, 8 negatives, deduplicated | both benchmark CSVs | `OUT_DIR/direction_set.csv` |
| 2 | Direction AUROC per scorer with a bootstrap CI - overall, singles against multi, by format | the structure, the scorers | `OUT_DIR/direction` |
| 3 | The Chinery binary: `positive` against `negative_clean`, overall and within each substitution count | the negatives CSV | `OUT_DIR/chinery` |
| 4 | Manufactured negatives as a floor check: hotspot alanine, implausible substitutions, non-interface decoys, `wt_argmax_rate` | the structure | `OUT_DIR/manufactured` |
| 5 | The verdict table, one row per scorer | Parts 2 to 4 | `OUT_DIR/verdict.csv` |

Parts 0 and 1 and every table in Parts 2 to 5 are cheap. The scoring cells need
Env B, a FoldX licence or a container, and Part 3 is the expensive one: 253
candidates for the clean split and 687 once the caveated negatives are in.

## What this notebook is not

**It is not a magnitude calibration.** Nothing here reports how much better a
positive was. Part 1 says why at length: three formats and three assays, and
only the sign survives pooling.

**It is not a replacement for the ranking notebooks.** A scorer can pass
direction and still rank badly inside the improved set, which is the question
`03_chinery_size_calibration.ipynb` asks and answers in the negative past four
substitutions. Direction is the weaker claim. It is here because the weaker
claim had never been checked, and because a scorer that fails it fails
everything downstream of it.

**It is not powered.** Twenty-one against eight is a confidence interval roughly
0.4 wide, so the interval is reported everywhere below and the point estimate
nowhere on its own. Several scorers are tested against the same small panels, so
Part 5 states how many were tested beside whichever one wins.

In [ ]:
# --- parameters -------------------------------------------------------------

EVIDENCE = "data/benchmarks/trastuzumab_mutation_evidence.csv"
NEGATIVES = "data/benchmarks/trastuzumab_negatives.csv"

# --- Part 1: the labelled direction set -------------------------------------

# Positives. The ADAPT panel's improvers, selected by their own measurement
# rather than by a hand-written list: the 2017 Fab study, the metric that
# carries a fold change, and a fold change above 1. On the file as of
# 2026-11-03 that is 20 of the 22 ADAPT variants; the other two are L:S50F and
# L:S50W, which are negatives and arrive from the other file.
ADAPT_STUDY = "adapt2017"
ADAPT_METRIC = "affinity_fold_improvement"
ADAPT_IMPROVED_ABOVE = 1.0

# Moon 2016's D98W, the one improver outside ADAPT with a clean matched parent
# on the same run (0.11 nM against 0.48 nM). Named by evidence id, because "the
# moon2016 row with a KD" is seven variants and a control.
MOON_POSITIVE = "TZM-019"

EXPECTED_POSITIVES = 21
EXPECTED_POSITIVE_SIZES = {1: 10, 2: 7, 3: 4}

# Negatives. Selected by label, not by block: `negative_affinity` is the measured
# affinity losses, and it appears in both blocks of the negatives file - 8
# variants in the literature block and 2 more in the chinery block, carried over
# from ADAPT with no evidence id, which is the duplication the dedup step is
# about. `negative_developability` (heterogeneous on purification, too low to
# characterise) and `negative_function` (reduced ADCC, reduced cell binding) are
# real failures and are not a binding direction, so they are dropped and named.
# NEGATIVE_SOURCE is the literature block, which is where that report comes from.
NEGATIVE_LABEL = "negative_affinity"
NEGATIVE_SOURCE = "clearly_negative_variants"

EXPECTED_NEGATIVES = 8
EXPECTED_NEGATIVE_SIZES = {1: 3, 2: 2, 3: 2, 7: 1}

# L:S50F and L:S50W are in both files, and twice in the negatives file: once in
# the literature block with an evidence id, once in the chinery block carried
# over from ADAPT with none. The assertion in Part 1 is that deduplication finds
# exactly these two collisions - a third would mean the files have drifted.
EXPECTED_DUPLICATES = ("L:S50F", "L:S50W")

# The weak negative, named as a parameter so that the sensitivity check in
# Part 2 is a stated prediction and not a sentence written afterwards. The
# source calls the affinity "similar": 4.8 +/- 0.7 nM against 2.0 +/- 0.4 nM,
# no significance test.
WEAK_NEGATIVE = "L:R66G"

# --- the structure ----------------------------------------------------------

# The local 1N8Z: trastuzumab Fab bound to HER2 domain IV. The heavy chain is
# 'B', not 'H', and both benchmark files write H and L, so CHAIN_MAP is the one
# place that is translated and Part 2 asserts it against the coordinates.
PDB_PATH = None                    # None -> the candidates searched in Part 2
HEAVY_CHAIN = "B"
LIGHT_CHAIN = "A"
ANTIGEN_CHAIN = "C"
CHAIN_MAP = {"H": HEAVY_CHAIN, "L": LIGHT_CHAIN}

OUT_DIR = "runs/negative_controls"

# --- the scorers ------------------------------------------------------------

# Split by what each can answer, because the split decides which negatives it is
# graded on. A matrix scorer answers one substitution from a per-position lookup
# and returns an empty cell for a candidate with several - never a sum - so it
# sees 10 of the 21 positives and 3 of the 8 negatives. The multi-capable
# scorers see all of both.
MATRIX_MODELS = ("antifold", "ablang2", "esm1v", "esm2_650m", "antiberty", "consensus")
MULTI_MODELS = ("antifold_seq", "foldx", "gearbind")

# Env B's interpreter, the one that can `import antifold`. None falls back to
# $DESIGN_PIPELINE_ANTIFOLD, which is what the adapter reads anyway.
ANTIFOLD_PYTHON = None
ANTIFOLD_SRC = None                # None -> $ANTIFOLD_SRC; the matrix route needs it
ANTIFOLD_WEIGHTS = None            # a path, to reuse the 567 MB download
ANTIFOLD_DEVICE = "cpu"

# FoldX comes out of the environment: FOLDX_BINARY, and DESIGN_PIPELINE_FOLDX_PDB
# pointing at the repaired, trimmed Fv plus antigen domain the alanine scan ran
# on (its residue numbers must still be 1N8Z's).
FOLDX_PYTHON = None                # None -> $DESIGN_PIPELINE_FOLDX
FOLDX_WORKERS = None
FOLDX_RUNS = None

GEARBIND_IMAGE = None              # no published image; None -> reported unavailable

PLM_DEVICE = None                  # None -> the adapter's own default

# --- Part 2: the statistic --------------------------------------------------

BOOTSTRAP = 10_000                 # resamples for the CI
BOOTSTRAP_SEED = 20261103
CI = 0.95
MIN_POSITIVES = 3                  # below this, AUROC is NaN rather than a number
MIN_NEGATIVES = 2

# A permutation test is exact when the split is small enough to enumerate every
# assignment of the labels, and Monte Carlo above that. C(29,8) is 4.3 million,
# so the direction set is sampled; C(11,2) is 55, so the old test is exact.
EXACT_PERMUTATION_MAX = 200_000

# The old two-negative test, for comparison. Eleven ADAPT singles of which two
# were measured worse: the smallest p a one-sided exact rank test on that split
# can reach is 1/C(11,2), whatever the scorer does.
OLD_TEST_N = 11
OLD_TEST_NEGATIVES = 2

# --- Part 3: the Chinery binary ---------------------------------------------

CHINERY_STUDY = "chinery2026"
CHINERY_POSITIVE = "positive"              # fitted_KD, 156 wells
CHINERY_CLEAN = "negative_clean"           # bound_unfittable (I.C.), 97 wells
CHINERY_CAVEATED = "negative_caveated"     # source_NA, 434 wells
CHINERY_EXPECTED = {"positive": 156, "negative_clean": 97, "negative_caveated": 434}
CHINERY_SIZES = tuple(range(3, 11))        # the within-size AUROCs reported

# The caveated negatives are 434 more candidates. With FoldX that is the
# difference between an overnight run and a week. Left False, Part 3 reports the
# clean split and says the second table was not computed rather than guessing it.
SCORE_CAVEATED = True

# --- Part 4: the manufactured negatives -------------------------------------

# The ten FoldX alanine-scan hotspots live in design_pipeline.hotspots, which
# holds the numbers quoted in docs/track_a_filtering.md. A hotspot is a position
# that loses at least HOTSPOT_DDG kcal/mol of binding on alanine.
HOTSPOT_DDG = 1.0

# The variable domains, which is where every manufactured variant is allowed to
# land. 1N8Z's chains are a whole light chain (214 residues) and a heavy Fd
# (220), so most of each chain is a constant domain that this pipeline never
# designs in and that no scorer here is calibrated on. The bounds are the ones
# design_pipeline.hotspots documents: VH 1 to 120, VL 1 to 107.
VARIABLE_DOMAIN = {HEAVY_CHAIN: 120, LIGHT_CHAIN: 107}

BURIED_BELOW = 0.15                # relative SASA under which a side chain is buried
SURFACE_ABOVE = 0.40               # relative SASA over which a side chain is exposed
BULKY = "FWYLIMV"                  # the side chains glycine is a real loss for
SALT_BRIDGE_CUTOFF = 4.0           # angstrom, carboxyl oxygen to basic nitrogen
PARATOPE_CUTOFF = 5.0              # angstrom, the interface definition

DECOY_POSITIONS = 20               # non-interface surface positions sampled for decoys
DECOY_SEED = 20261103

# Each structure-derived class is capped and sampled with DECOY_SEED, because a
# Fv has more mid-strand positions and more salt bridges than a per-variant
# scorer can afford. The cap is a budget, not a judgement: raise it and the
# failure rates get tighter intervals at linear cost in FoldX hours.
MAX_PER_CLASS = 20

# What counts as a failure. `score_<model>` is higher-is-better-binding and zero
# is the wild type by construction for every scorer here, so a manufactured
# negative scored above zero is the scorer calling a deliberate break an
# improvement. The harder criterion is scoring above the median of the real
# positives from Part 1, which is what a shortlist cut would act on.
FAILURE_ABOVE = 0.0

MAX_VARIANTS = 1_000               # the cap score_variants applies to per-variant scorers
CACHE = True

In [ ]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from design_pipeline import hotspots, metrics, preflight, structures
from design_pipeline.api import score_variants
from design_pipeline.errors import PipelineError
from design_pipeline.matrices import IndexMismatchError
from design_pipeline.structures import AMINO_ACIDS, Complex, chain_sequence
from design_pipeline.variants import enumerate_variants, make_id

diagnostics = metrics.diagnostics          # design_pipeline._diagnostics, loaded on use

pd.set_option("display.width", 185)
pd.set_option("display.max_columns", 50)

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

ALL_MODELS = tuple(MATRIX_MODELS) + tuple(MULTI_MODELS)
print(preflight.render(preflight.rows(ALL_MODELS)))
for note in preflight.notes(ALL_MODELS):
    print(f"\n{note}")
print(f"\nmatrix scorers, one substitution only: {', '.join(MATRIX_MODELS)}")
print(f"multi-capable scorers:                 {', '.join(MULTI_MODELS)}")
print("\nA scorer with no route here is reported as missing and costs you its row.")
print("Nothing below substitutes one scorer for another and nothing fills a")
print("column with zeros: an unscored cell stays empty and is counted as unscored.")

## Part 1 - the labelled direction set

Two files, one label. `trastuzumab_mutation_evidence.csv` is one row per
*measurement*, so a variant owns several rows. `trastuzumab_negatives.csv` is
one row per *substitution*, so a triple owns three. Neither is a variant table,
and collapsing each to one row per variant is the first thing that happens here.

**Positives (21).** The 20 ADAPT 2017 Fab variants whose measured fold
improvement is above 1 - nine singles, seven doubles, four triples, built from
three sites: `H:D31` in CDR-H1, `H:D102` in CDR-H3 and `L:D28` in CDR-L1. The
ADAPT panel's fourth site, `L:S50`, appears only in its two losses, which is why
the positive half has three sites and 20 variants. Plus Moon 2016's `D98W`
(`H:D102W` in
mature-domain numbering), an IgG at 0.11 nM against a 0.48 nM wild type on the
same SPR run. Ten singles in all, then, which is what the matrix scorers get.

**Negatives (8).** The literature block of the negatives file, restricted to
measured affinity losses:

| variant | subs | format | what was measured |
|---|---|---|---|
| `L:S50F` | 1 | Fab | 0.3x the parent's affinity (ADAPT) |
| `L:S50W` | 1 | Fab | 0.5x the parent's affinity (ADAPT) |
| `L:R66G` | 1 | scFv | KD 4.8 +/- 0.7 nM against 2.0 +/- 0.4 nM |
| `L:V33L+L:R66G` | 2 | scFv | KD 4.9 nM against 2.0 nM |
| `L:V33I+L:R66G` | 2 | scFv | KD 5.2 nM against 2.0 nM |
| `AH12` | 3 | IgG | KD 1.12 nM against 0.48 nM |
| `AL07` | 3 | IgG | KD 0.79 nM against 0.48 nM |
| `A058` | 7 | IgG | KD 1.05 nM against 0.48 nM |

What gets dropped and why is printed below. The negatives file also carries
`negative_developability` - two triples heterogeneous on purification, one
double expressed too low to characterise - and `negative_function` - reduced
ADCC, reduced cell binding. Those are real failures and they are not a *binding*
direction. Grading an affinity scorer on a purification problem is grading it on
something it does not model.

**The deduplication.** `L:S50F` and `L:S50W` are in both files, and twice in the
negatives file: once in the literature block with evidence ids `TZM-062` and
`TZM-063`, once in the chinery block carried over from ADAPT with no id. The
positive selection excludes them by its own fold-change filter, so they can only
arrive as negatives, which is what they are. The assertion below is that exactly
these two collide and nothing else does - the two files are curated by hand, and
this is where a drift between them would surface.

In [ ]:
# The benchmark's own notation, kept separate from `variant_id`: the first is
# (H:/L:, mature-domain 1-based), the second is this package's, over the chains
# 1N8Z actually has. The same three helpers as 03_adapt_calibration.ipynb and
# 03_chinery_size_calibration.ipynb, so the three notebooks parse one way.
TOKEN = re.compile(r"^(?P<chain>[A-Za-z0-9]+):(?P<wt>[A-Z])(?P<position>\d+)(?P<mut>[A-Z])$")


def parse_benchmark_variant(text):
    '''[(chain, position, wt, mut)] as the benchmark writes it.'''
    mutations = []
    for token in str(text).split(";"):
        match = TOKEN.match(token.strip())
        if match is None:
            raise ValueError(f"{text!r}: {token.strip()!r} is not a substitution like H:D31R")
        mutations.append((match["chain"], int(match["position"]), match["wt"], match["mut"]))
    if not mutations:
        raise ValueError(f"{text!r} names no substitution")
    return mutations


def remap(mutations):
    '''The same mutations over 1N8Z's chain ids. An unmapped chain is an error.'''
    out = []
    for chain, position, wt, mut in mutations:
        if chain not in CHAIN_MAP:
            raise KeyError(f"chain {chain!r} is not in CHAIN_MAP {CHAIN_MAP}")
        out.append((CHAIN_MAP[chain], position, wt, mut))
    return out


def label_of(mutations):
    '''The benchmark-side label: H:D31R+H:D102F, sorted, chains as written.'''
    return "+".join(f"{c}:{w}{p}{m}" for c, p, w, m in sorted(mutations))


evidence = pd.read_csv(EVIDENCE, low_memory=False)
negatives_file = pd.read_csv(NEGATIVES)
print(f"{EVIDENCE}\n  {len(evidence):,} rows, one per measurement")
print(f"{NEGATIVES}\n  {len(negatives_file):,} rows, one per substitution")

for frame, path, needed in (
    (evidence, EVIDENCE, ["evidence_id", "study", "variant", "format", "metric", "value",
                          "substitutions_mature_domain_1based", "outcome", "assay",
                          "parent_value", "uncertainty", "notes"]),
    (negatives_file, NEGATIVES, ["variant", "evidence_id", "label_class", "label_reason",
                                 "n_mutations", "chain", "position", "wt", "mut", "format",
                                 "study", "source_file"]),
):
    absent = [c for c in needed if c not in frame.columns]
    if absent:
        raise ValueError(f"{path} is missing columns this notebook reads: {absent}")

print(f"\n{Path(NEGATIVES).name} by block and label (distinct variants):")
print(negatives_file.groupby(["source_file", "label_class"])["variant"]
      .nunique().to_frame("variants").to_string())

In [ ]:
# --- the positives ----------------------------------------------------------
# Selected by the measurement, not by a list of names, so a row that goes in
# comes with the number that put it there.
adapt = evidence[(evidence["study"] == ADAPT_STUDY) & (evidence["metric"] == ADAPT_METRIC)]
fold = pd.to_numeric(adapt["value"], errors="coerce")
if fold.isna().any():
    raise ValueError(f"{ADAPT_METRIC} rows with a non-numeric value: "
                     f"{sorted(adapt.loc[fold.isna(), 'variant'])}")
improved = adapt[fold > ADAPT_IMPROVED_ABOVE]
worsened = adapt[fold <= ADAPT_IMPROVED_ABOVE]

print(f"{ADAPT_STUDY} / {ADAPT_METRIC}: {len(adapt)} variants, "
      f"{len(improved)} above {ADAPT_IMPROVED_ABOVE}x, {len(worsened)} at or below")
print(f"  the {len(worsened)} at or below are {', '.join(sorted(worsened['variant']))}"
      f" - negatives, and they arrive from {Path(NEGATIVES).name}, not from here")

moon = evidence[evidence["evidence_id"] == MOON_POSITIVE]
if len(moon) != 1:
    raise ValueError(f"{MOON_POSITIVE} selects {len(moon)} rows in {EVIDENCE}, expected 1")
moon_row = moon.iloc[0]
print(f"\n{MOON_POSITIVE}: {moon_row['study']} {moon_row['variant']} "
      f"= {moon_row['substitutions_mature_domain_1based']}, {moon_row['format']}, "
      f"{moon_row['assay']}")
print(f"  KD {moon_row['value']} nM against a parent at {moon_row['parent_value']} nM "
      f"-> {moon_row['outcome']}")

records = []
for row in improved.itertuples():
    mutations = parse_benchmark_variant(row.substitutions_mature_domain_1based)
    records.append({
        "evidence_id": row.evidence_id, "source": ADAPT_STUDY,
        "label": label_of(mutations), "format": row.format,
        "direction": 1, "n_mutations": len(mutations),
        "evidence": f"{row.value}x fold improvement", "assay": row.assay,
        "mutations": tuple(sorted(remap(mutations))),
    })
moon_mutations = parse_benchmark_variant(moon_row["substitutions_mature_domain_1based"])
records.append({
    "evidence_id": moon_row["evidence_id"], "source": moon_row["study"],
    "label": label_of(moon_mutations), "format": moon_row["format"],
    "direction": 1, "n_mutations": len(moon_mutations),
    "evidence": f"KD {moon_row['value']} vs parent {moon_row['parent_value']} nM",
    "assay": moon_row["assay"],
    "mutations": tuple(sorted(remap(moon_mutations))),
})
positives = pd.DataFrame(records)

if len(positives) != EXPECTED_POSITIVES:
    raise ValueError(f"expected {EXPECTED_POSITIVES} positives, built {len(positives)}")
sizes = positives["n_mutations"].value_counts().to_dict()
if sizes != EXPECTED_POSITIVE_SIZES:
    raise ValueError(f"expected {EXPECTED_POSITIVE_SIZES} positives by size, got {sizes}")
print(f"\n{len(positives)} positives: "
      + ", ".join(f"{n} with {k} substitution(s)" for k, n in sorted(sizes.items())))

In [ ]:
# --- the negatives ----------------------------------------------------------
# Selected by label and not by block, because `negative_affinity` is in both
# blocks: the literature block has the eight with evidence ids, and the chinery
# block carries two more over from ADAPT with no id at all. Both are taken here
# so that the deduplication in the next cell has something to do - filtering to
# one block would hide the collision instead of printing it.
#
# One row per substitution, so a variant is a group, and the group key is
# (variant, study) rather than evidence_id for the same reason: two of the
# groups have no id.
affinity = negatives_file[negatives_file["label_class"] == NEGATIVE_LABEL]
literature = negatives_file[negatives_file["source_file"] == NEGATIVE_SOURCE]
other = literature[literature["label_class"] != NEGATIVE_LABEL]

print(f"{NEGATIVE_LABEL} across the whole file: {affinity['variant'].nunique()} "
      f"distinct variant names in {len(affinity)} substitution rows, by block:")
print(affinity.groupby(["source_file", "study"])["variant"].nunique()
      .to_frame("variants").to_string())
print(f"\n{NEGATIVE_SOURCE}: {literature['variant'].nunique()} variants, of which")
print(f"  {literature[literature['label_class'] == NEGATIVE_LABEL]['variant'].nunique()}"
      f" are {NEGATIVE_LABEL} and go in")
print(f"  {other['variant'].nunique()} are not a binding direction and are dropped:")
for (variant, label, reason), _ in other.groupby(
    ["variant", "label_class", "label_reason"], sort=True
):
    print(f"    {variant:<30}{label:<26}{reason}")
print("  A purification or ADCC failure is a real failure. It is not a measured")
print("  change in affinity, and an affinity scorer graded on it is graded on")
print("  something it does not model.")

records = []
for (variant, study), group in affinity.groupby(["variant", "study"], sort=True):
    mutations = [
        (str(r.chain), int(r.position), str(r.wt), str(r.mut)) for r in group.itertuples()
    ]
    declared = sorted(set(int(n) for n in group["n_mutations"]))
    if declared != [len(mutations)]:
        raise ValueError(
            f"{variant}: {len(mutations)} substitution rows but n_mutations says "
            f"{declared}. The file's own count disagrees with its own rows, so one "
            f"of the two is wrong and the sizes below cannot be trusted."
        )
    formats = sorted(set(group["format"]))
    if len(formats) != 1:
        raise ValueError(f"{variant}: more than one format on one variant: {formats}")
    ids = sorted({i for i in group["evidence_id"] if isinstance(i, str) and i})
    records.append({
        "evidence_id": ids[0] if ids else "",
        "source": study, "well": variant, "label": label_of(mutations),
        "format": formats[0], "direction": 0, "n_mutations": len(mutations),
        "evidence": sorted(set(group["label_reason"]))[0], "assay": "",
        "mutations": tuple(sorted(remap(mutations))),
    })
raw_negatives = pd.DataFrame(records)
print(f"\n{len(raw_negatives)} {NEGATIVE_LABEL} variants before deduplication")
print(raw_negatives[["well", "label", "n_mutations", "format", "source",
                     "evidence_id", "evidence"]].to_string(index=False))

In [ ]:
# --- deduplicate, on the mutation set and nothing else ----------------------
# `label` is the key because it is derived from the substitutions, so two rows
# naming the same variant in different notation still collide. The assertion is
# that the collisions are exactly EXPECTED_DUPLICATES.
collisions = (
    raw_negatives.groupby("label").filter(lambda g: len(g) > 1).sort_values(["label", "source"])
)
found = tuple(sorted(collisions["label"].unique()))
print(f"labels appearing more than once: {', '.join(found) if found else '(none)'}")
if not collisions.empty:
    print(collisions[["label", "source", "well", "evidence_id", "evidence", "format"]]
          .to_string(index=False))
if found != tuple(sorted(EXPECTED_DUPLICATES)):
    raise ValueError(
        f"expected the duplicates to be exactly {sorted(EXPECTED_DUPLICATES)}, found "
        f"{list(found)}. Both files are curated by hand and this is where a drift "
        f"between them shows up; reconcile the two before reading any AUROC below."
    )

# Keep the copy carrying an evidence id, so the surviving row points back at a
# measurement rather than at a label carried over from another study.
raw_negatives = raw_negatives.assign(has_id=raw_negatives["evidence_id"].astype(bool))
negatives = (
    raw_negatives.sort_values(["label", "has_id"], ascending=[True, False])
    .drop_duplicates("label", keep="first")
    .drop(columns=["has_id", "well"])
    .reset_index(drop=True)
)
if len(negatives) != EXPECTED_NEGATIVES:
    raise ValueError(f"expected {EXPECTED_NEGATIVES} negatives, kept {len(negatives)}")
sizes = negatives["n_mutations"].value_counts().to_dict()
if sizes != EXPECTED_NEGATIVE_SIZES:
    raise ValueError(f"expected {EXPECTED_NEGATIVE_SIZES} negatives by size, got {sizes}")
print(f"\n{len(negatives)} negatives after deduplication: "
      + ", ".join(f"{n} with {k} substitution(s)" for k, n in sorted(sizes.items())))

# The cross-check deduplication cannot make: a label must not be in both halves.
# If one paper measured a variant better and another measured it worse, that is a
# finding, not two rows to average.
overlap = sorted(set(positives["label"]) & set(negatives["label"]))
if overlap:
    raise ValueError(
        f"{len(overlap)} label(s) are both positive and negative: {overlap}. Two "
        f"measurements disagree about the direction and this notebook cannot decide "
        f"which to believe."
    )
print(f"no label is in both halves ({positives['label'].nunique()} positive labels, "
      f"{negatives['label'].nunique()} negative)")

direction = (
    pd.concat([positives, negatives], ignore_index=True)
    .assign(variant_id=lambda f: [make_id(m) for m in f["mutations"]])
    .assign(multi=lambda f: f["n_mutations"] > 1)
    .sort_values(["direction", "n_mutations", "label"], ascending=[False, True, True])
    .reset_index(drop=True)
)
if direction["variant_id"].duplicated().any():
    raise ValueError("a variant_id appears twice after the concat")

In [ ]:
# --- the counts per cell, which is the point of printing anything at all ----
print(f"the direction set: {len(direction)} variants, "
      f"{int(direction['direction'].sum())} improved, "
      f"{int((1 - direction['direction']).sum())} worsened\n")


def per_cell(column, name):
    '''improved / worsened / total, one row per value of `column`.'''
    table = pd.crosstab(
        direction[column], direction["direction"].map({1: "improved", 0: "worsened"})
    ).reindex(columns=["improved", "worsened"], fill_value=0)
    table.index.name = name
    table["total"] = table.sum(axis=1)
    table.loc["ALL"] = table.sum()
    return table


for column, name in (("n_mutations", "substitutions"), ("format", "format"),
                     ("source", "study"), ("multi", "several subs")):
    print(per_cell(column, name).to_string(), "\n")

print("Read the format table twice. The scFv rows are negatives only and the IgG")
print("rows are one positive against three negatives, so a by-format AUROC exists")
print("for Fab and for nothing else - which Part 2 reports rather than papers over.")
print()
print(direction[["label", "direction", "n_mutations", "format", "source", "evidence"]]
      .assign(direction=lambda f: f["direction"].map({1: "improved", 0: "worsened"}))
      .to_string(index=False))

# The positions, because they are reused and reuse is non-independence.
sites = {}
for row in direction.itertuples():
    for chain, position, wt, _mut in row.mutations:
        key = (chain, position, wt)
        entry = sites.setdefault(key, {"improved": 0, "worsened": 0})
        entry["improved" if row.direction == 1 else "worsened"] += 1
print(f"\n{len(sites)} distinct sites over {len(direction)} variants:")
print(f"{'site (1N8Z)':<14}{'in improved':>13}{'in worsened':>13}")
print("-" * 40)
for (chain, position, wt), counts in sorted(sites.items()):
    print(f"{f'{chain}:{wt}{position}':<14}{counts['improved']:>13}{counts['worsened']:>13}")
print("-" * 40)
positive_sites = {k for k, v in sites.items() if v["improved"]}
negative_sites = {k for k, v in sites.items() if v["worsened"]}
print(f"{len(positive_sites)} site(s) carry all {int(direction['direction'].sum())} "
      f"improvers and {len(negative_sites)} carry all "
      f"{int((1 - direction['direction']).sum())} losses; "
      f"{len(positive_sites & negative_sites)} site(s) are in both.")
print(f"{direction['source'].nunique()} studies in all. The bootstrap in Part 2")
print("resamples variants, not sites and not studies, so it does not know about")
print("any of this and the interval it prints is therefore the optimistic one.")

path = out_dir / "direction_set.csv"
direction.drop(columns=["mutations"]).to_csv(path, index=False)
print(f"\n-> {path}")

### Direction only, never KD, and the table above is the reason

Three formats and three assays. ADAPT is **Fab**, monovalent, SPR, reported as a
fold improvement against a batch-matched parent run alongside. Moon is **IgG**,
two Fab arms, SPR, an absolute KD against a wild-type well on the same run.
Okazaki is **scFv**, a different construct again, with its own parent and a
melting temperature that the paper reports moving.

**The sign is comparable across all three. The magnitude is not.** A 2.8-fold
Fab improvement and a 4.4-fold IgG improvement are not 2.8 and 4.4 on one scale:
the IgG number carries an avidity term the Fab number does not have, the scFv
number carries a construct whose stability changed, and the SPR runs were
normalised against different parents. So **nothing in this notebook pools a KD,
a fold change or a ddG across those rows, and nothing correlates against one.**
The only thing asked of a label is which side of its own parent the variant
landed on, which is the one property all three assays agree on how to report.

That is also why the statistic is AUROC and not Spearman. Spearman over the
pooled set would be ranking a Fab fold change against an IgG KD ratio and
calling the result a calibration. AUROC only ever compares pairs with opposite
labels, so it never puts two magnitudes on one axis at all.

**Three things about these eight negatives, out loud.**

* **`L:R66G` is a weak negative.** The source describes the affinity as
  *similar* and reports **4.8 +/- 0.7 nM against 2.0 +/- 0.4 nM with no
  significance test**; the curator kept the numerical increase and said so in
  the row's `notes`. It is in the set because dropping it would leave two single
  negatives, which is the hole this notebook exists to fill. It is flagged here
  because a 2.4-fold shift with overlapping stated uncertainties and no test is
  not the evidence that ADAPT's 0.3x is. Part 2 reports every AUROC again with
  it dropped, so the weight it carries is visible rather than argued about.
* **Three of the eight share one substitution.** `L:R66G` is the whole of one
  negative and half of the other two, so the negative half is nearer five
  independent observations than eight. The bootstrap resamples variants, so it
  does not know this; the interval it prints is the optimistic one.
* **`A058` contains `H:D102W`, which is also the whole of a positive.** Moon's
  `D98W` is `H:D102W` and improved affinity 4.4-fold. `A058` is `H:D102W` plus
  six more substitutions and worsened it. Moon's own `AH06` -
  `H:D102W+H:M107F+H:D108A+H:Y109L`, 8-fold improved, deliberately not in this
  set because the set is the one specified - differs from `A058` only by three VL
  substitutions. So one substitution is the best single in the panel, a
  component of an 8-fold improvement, and a component of a loss, depending on
  what sits beside it. **No scorer that treats substitutions as independent can
  get all three right**, and that is a ceiling on the matrix scorers that no
  amount of extra data moves.

## Part 2 - direction AUROC per scorer

The numbering gate first, then one scoring call per scorer, then the statistic.

**The gate.** Both files number their substitutions into the mature domain,
1-based; a `variant_id` here is 1-based into *the chain as given in the
structure*. Those agree only if 1N8Z's chains start at domain residue 1 with no
gaps, which is not a thing to assume: a set numbered in Kabat or IMGT would land
every substitution a few residues off and still score perfectly cleanly, and the
AUROC would be a real number about the wrong residues. The check is the simple
one - the residue each file says is wild type has to be the residue 1N8Z has at
that chain and position - run over every site in the set, and then made a second
time by `enumerate_variants`, which re-checks on the way in. Nothing after it
runs if it fails.

**The statistic.** AUROC computed from ranks, so ties count half, which matters:
a matrix scorer gives the same number to every candidate at a position it has no
opinion about. Three readings of it, in order:

* **Overall**, per scorer, over whatever that scorer could answer.
* **Singles against multi-mutation.** The matrix scorers see 10 positives and 3
  negatives and nothing else; the multi-capable scorers see all 21 and all 8.
  These are not the same test and the numbers are not comparable across the
  split, which is why they are two columns and not one.
* **By format.** If a scorer's errors cluster in one format, the pooled number
  is reporting an assay artifact. Part 1's format table says in advance that
  only Fab has both labels, so this reading is mostly a check that the IgG and
  scFv rows are not quietly carrying the result.

**The interval.** A stratified bootstrap - positives resampled from positives,
negatives from negatives, `BOOTSTRAP` times - and the percentile interval. With
21 against 8 that interval is about 0.4 wide, and it is printed next to every
point estimate for exactly that reason. It is also optimistic: it resamples
variants, and Part 1 showed the variants share fifteen sites and three studies -
with all 21 improvers built from three of those sites.

**And the old test, for comparison.** Before these negatives existed the whole
of the direction evidence was 11 ADAPT singles of which 2 were measured worse.
The smallest p a one-sided exact rank test on that split can reach is
**1/C(11,2) = 1/55 = 0.018**, and that is the *best possible* outcome - a scorer
that ranked both losses dead last could not do better. It is reported below
beside the real p values so that what the extra six negatives bought is a
number: the floor moves from 0.018 to 1/C(29,8), which is 2.3e-7.

In [ ]:
CANDIDATES = (
    Path("data/structures/1N8Z.pdb"),
    Path("data/1N8Z.pdb"),
    Path.home() / ".cache" / "design_pipeline" / "pdb" / "1n8z.pdb",
)


def resolve_pdb():
    if PDB_PATH is not None:
        path = Path(PDB_PATH).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f"PDB_PATH does not exist: {path}")
        return path
    for candidate in CANDIDATES:
        if candidate.is_file():
            return candidate
    looked = "\n".join(f"  {c}" for c in CANDIDATES)
    raise FileNotFoundError(
        f"no local 1N8Z found. Looked in:\n{looked}\nSet PDB_PATH to the copy you "
        f"mean to use. Nothing here downloads: a structure that arrived by another "
        f"route is a different structure, and every position below is an index into "
        f"this one file."
    )


pdb = resolve_pdb()
chains_present = structures.chain_ids(pdb)
roles = {"heavy": HEAVY_CHAIN, "light": LIGHT_CHAIN, "antigen": ANTIGEN_CHAIN}
unknown = {role: chain for role, chain in roles.items() if chain not in chains_present}
if unknown:
    raise KeyError(f"chains named in the parameters that {pdb} does not have: "
                   f"{unknown}. Present: {chains_present}")
sequences = {chain: chain_sequence(pdb, chain) for chain in chains_present}
residue_ids = {chain: structures.chain_residue_ids(pdb, chain) for chain in chains_present}

print(f"structure: {pdb}")
print(f"{'chain':<7}{'role':<9}{'len':>5}{'first':>8}{'last':>8}{'labels 1..n':>14}")
print("-" * 52)
role_of = {v: k for k, v in roles.items()}
contiguous = {}
for chain in chains_present:
    labels = residue_ids[chain]
    numeric = all(label.isdigit() for label in labels)
    runs = numeric and [int(l) for l in labels] == list(range(1, len(labels) + 1))
    contiguous[chain] = runs
    print(f"{chain:<7}{role_of.get(chain, '-'):<9}{len(sequences[chain]):>5}"
          f"{labels[0]:>8}{labels[-1]:>8}{('yes' if runs else 'NO'):>14}")

# --- the gate: every site in the direction set, against the coordinates -----
sites = {}
for row in direction.itertuples():
    for chain, position, wt, _mut in row.mutations:
        sites.setdefault((chain, position, wt), []).append(row.label)

print(f"\n{'site (1N8Z)':<14}{'files say wt':>14}{'1N8Z has':>10}{'pdb label':>11}"
      f"{'variants':>10}  verdict")
print("-" * 73)
mismatches = []
for (chain, position, wt), used_by in sorted(sites.items()):
    sequence = sequences[chain]
    inside = 0 < position <= len(sequence)
    here = sequence[position - 1] if inside else "-"
    label = residue_ids[chain][position - 1] if inside else "-"
    ok = inside and here == wt
    if not ok:
        mismatches.append((chain, position, wt, here))
    print(f"{f'{chain}:{position}':<14}{wt:>14}{here:>10}{label:>11}{len(used_by):>10}  "
          f"{'ok' if ok else 'MISMATCH'}")
if mismatches:
    raise IndexMismatchError(
        "the numbering does not line up with the structure: "
        + "; ".join(f"{c}:{p} the files say {w}, 1N8Z has {h}" for c, p, w, h in mismatches)
        + ". Every AUROC below would be a real number for the wrong residues. Check "
        "CHAIN_MAP and each block's numbering before going on."
    )
NUMBERING_OK = True
print(f"\n{len(sites)} sites, {len(direction)} variants, every wild-type residue agrees")
print(f"(1N8Z's own labels run 1..n on chains {HEAVY_CHAIN} and {LIGHT_CHAIN}: "
      f"{contiguous[HEAVY_CHAIN] and contiguous[LIGHT_CHAIN]}, so the files' "
      f"mature-domain positions and this package's are the same positions here)")

In [ ]:
# --- one scoring helper, used by Parts 2, 3 and 4 ---------------------------
# `score_variants` sends **kwargs to every model in a call, so these three want
# separate calls: AntiFold needs chain ids and a source folder, FoldX reads its
# binary from the environment, the language models take a device. One call per
# model, each joining its column into the table the previous one returned.
complex_ = Complex(
    paths=(pdb,),
    output_dir=out_dir,
    config={"model_name": "local_file", "source": str(pdb)},
)

for name, setting in (("FOLDX_RUNS", FOLDX_RUNS), ("FOLDX_WORKERS", FOLDX_WORKERS)):
    if setting is not None:
        os.environ[name] = str(setting)

if GEARBIND_IMAGE is not None:
    from dataclasses import replace

    from design_pipeline import REGISTRY
    from design_pipeline.adapters import _scorers

    REGISTRY["gearbind"] = replace(
        _scorers.gearbind, tool=_scorers.matrix_container(GEARBIND_IMAGE)
    )
    print(f"gearbind -> {GEARBIND_IMAGE}")

MODEL_KWARGS = {
    "antifold": {"executable": ANTIFOLD_PYTHON, "heavy": HEAVY_CHAIN,
                 "light": LIGHT_CHAIN, "antifold_src": ANTIFOLD_SRC},
    "antifold_seq": {"executable": ANTIFOLD_PYTHON, "heavy": HEAVY_CHAIN,
                     "light": LIGHT_CHAIN, "device": ANTIFOLD_DEVICE,
                     "weights": ANTIFOLD_WEIGHTS},
    "foldx": {"executable": FOLDX_PYTHON},
    "ablang2": {"device": PLM_DEVICE, "antibody_chains": [HEAVY_CHAIN, LIGHT_CHAIN]},
    "esm1v": {"device": PLM_DEVICE, "antibody_chains": [HEAVY_CHAIN, LIGHT_CHAIN]},
    "esm2_650m": {"device": PLM_DEVICE, "antibody_chains": [HEAVY_CHAIN, LIGHT_CHAIN]},
    "antiberty": {"device": PLM_DEVICE, "antibody_chains": [HEAVY_CHAIN, LIGHT_CHAIN]},
}


def score_set(table, models, where, label):
    '''(table, ran, skipped) with one `score_<model>` column per scorer that ran.

    A scorer that cannot run here is recorded with the reason the package gave
    and the rest carry on: a missing FoldX licence should cost you the FoldX row
    of the verdict table, not the other eight. A scorer that runs and answers
    nothing is also a skip, because an all-empty column is not a result.
    '''
    ran, skipped = [], {}
    for model in models:
        kwargs = {k: v for k, v in MODEL_KWARGS.get(model, {}).items() if v is not None}
        try:
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", message="anarci is not installed")
                table = score_variants(
                    complex_, table, where, models=[model], structure=pdb,
                    max_variants=MAX_VARIANTS, cache=CACHE, **kwargs,
                )
        except PipelineError as exc:
            skipped[model] = f"{type(exc).__name__}: {exc}"
            print(f"  {model:<14} SKIPPED  {type(exc).__name__}: "
                  f"{str(exc).splitlines()[0]}")
            continue
        column = f"score_{model}"
        one = table.frame.drop_duplicates("variant_id")
        answered = int(one[column].notna().sum()) if column in one.columns else 0
        if answered == 0:
            skipped[model] = "ran, but scored nothing"
            print(f"  {model:<14} ran and scored nothing, so treated as skipped")
            continue
        ran.append(model)
        print(f"  {model:<14} {answered:>5} of {table.candidates} candidates scored")
    print(f"{label}: scorers that ran: {', '.join(ran) if ran else 'none'}")
    return table, ran, skipped


if not NUMBERING_OK:
    raise RuntimeError("Part 2's numbering gate did not pass")

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="anarci is not installed")
    direction_table = enumerate_variants(
        complex_, HEAVY_CHAIN, out_dir / "direction" / "variants",
        only=direction["variant_id"].tolist(), structure=pdb,
    )
print(f"{direction_table.rows} rows, {direction_table.candidates} candidates "
      f"-> {direction_table.path}")
if direction_table.candidates != len(direction):
    raise ValueError(f"{direction_table.candidates} candidates for {len(direction)} variants")

print("\nscoring the direction set:")
direction_scored, direction_ran, direction_skipped = score_set(
    direction_table, ALL_MODELS, out_dir / "direction" / "scores", "direction set"
)
if not direction_ran:
    raise RuntimeError(
        "no scorer ran, so there is no AUROC to report. Part 0's table says what "
        "each one needs; the cheapest to satisfy are the language models, which "
        "install into this environment with the `plm` extra."
    )

In [ ]:
# --- AUROC, its bootstrap interval, and a permutation p ---------------------
# AUROC from averaged ranks, which is what makes ties count half. That is not a
# detail here: a matrix scorer returns the same number at every position it has
# no opinion about, and counting those pairs as wins would be a free half of the
# statistic.
RNG = np.random.default_rng(BOOTSTRAP_SEED)


def auroc(scores, labels):
    '''(auroc, n_pos, n_neg) over the finite scores. NaN when the cells are too thin.

    NaN rather than 0.5, because 0.5 is a claim - "this scorer is a coin" - and
    the absence of enough data to make a claim is not that claim.
    '''
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    usable = np.isfinite(scores)
    s, y = scores[usable], labels[usable]
    n_pos, n_neg = int((y == 1).sum()), int((y == 0).sum())
    if n_pos < MIN_POSITIVES or n_neg < MIN_NEGATIVES:
        return float("nan"), n_pos, n_neg
    ranks = metrics.rankdata(s)
    u = float(ranks[y == 1].sum()) - n_pos * (n_pos + 1) / 2.0
    return u / float(n_pos * n_neg), n_pos, n_neg


def auroc_ci(scores, labels, resamples=None):
    '''Percentile CI from a stratified bootstrap: positives and negatives separately.

    Stratified because the class sizes are the design of the panel, not a draw
    from anything - resampling the pooled set would sometimes produce a draw
    with no negatives at all, and the interval would then be an interval over a
    different experiment.
    '''
    resamples = BOOTSTRAP if resamples is None else resamples
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    usable = np.isfinite(scores)
    s, y = scores[usable], labels[usable]
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    if len(pos) < MIN_POSITIVES or len(neg) < MIN_NEGATIVES:
        return float("nan"), float("nan")
    draws = np.empty(resamples, dtype=float)
    for i in range(resamples):
        take = np.concatenate([
            RNG.choice(pos, size=len(pos), replace=True),
            RNG.choice(neg, size=len(neg), replace=True),
        ])
        draws[i] = auroc(s[take], y[take])[0]
    draws = draws[np.isfinite(draws)]
    if not len(draws):
        return float("nan"), float("nan")
    low, high = (1.0 - CI) / 2.0, 1.0 - (1.0 - CI) / 2.0
    return float(np.quantile(draws, low)), float(np.quantile(draws, high))


def permutation_p(scores, labels, resamples=None):
    '''One-sided P(AUROC >= observed) under the null, and whether it was exact.

    Exact by enumeration when the number of label assignments is small enough,
    Monte Carlo above that, with the usual (hits + 1) / (draws + 1) so the
    reported p is never zero. `floor` is the smallest p this design can reach -
    1/C(n, n_neg) exact, 1/(resamples + 1) sampled - which is the number that
    makes a small panel's significance legible.
    '''
    from itertools import combinations
    from math import comb

    resamples = BOOTSTRAP if resamples is None else resamples
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    usable = np.isfinite(scores)
    s, y = scores[usable], labels[usable]
    n, n_neg = len(y), int((y == 0).sum())
    observed = auroc(s, y)[0]
    if observed != observed:
        return float("nan"), float("nan"), "too few"
    total = comb(n, n_neg)
    if total <= EXACT_PERMUTATION_MAX:
        hits = 0
        for picked in combinations(range(n), n_neg):
            shuffled = np.ones(n, dtype=int)
            shuffled[list(picked)] = 0
            if auroc(s, shuffled)[0] >= observed - 1e-12:
                hits += 1
        return hits / float(total), 1.0 / float(total), f"exact over {total:,}"
    hits = 0
    for _ in range(resamples):
        if auroc(s, RNG.permutation(y))[0] >= observed - 1e-12:
            hits += 1
    return (hits + 1) / float(resamples + 1), 1.0 / float(resamples + 1), \
        f"sampled, {resamples:,} draws"


def cell(value, width=0):
    text = "-" if value != value else f"{value:.3f}"
    return f"{text:>{width}}" if width else text


# The old test's ceiling, computed rather than quoted.
from math import comb

old_floor = 1.0 / comb(OLD_TEST_N, OLD_TEST_NEGATIVES)
new_floor = 1.0 / comb(len(direction), int((1 - direction["direction"]).sum()))
print(f"the old two-negative test: {OLD_TEST_N} ADAPT singles, "
      f"{OLD_TEST_NEGATIVES} of them measured worse")
print(f"  best possible p = 1/C({OLD_TEST_N},{OLD_TEST_NEGATIVES}) = "
      f"1/{comb(OLD_TEST_N, OLD_TEST_NEGATIVES)} = {old_floor:.4f}")
print(f"  a scorer that put both losses dead last could not beat that, so nothing")
print(f"  measured on it could ever have reached p < 0.01.")
print(f"\nthis set: {len(direction)} variants, "
      f"{int((1 - direction['direction']).sum())} of them measured worse")
print(f"  best possible p = 1/C({len(direction)},"
      f"{int((1 - direction['direction']).sum())}) = {new_floor:.2e}")
print(f"  the floor drops by a factor of {old_floor / new_floor:,.0f}. That is what")
print(f"  the extra six negatives bought, and it is a bound on what is provable,")
print(f"  not evidence that anything is true.")

In [ ]:
# --- the headline: direction AUROC, overall, with its interval --------------
labels = direction["direction"].to_numpy(dtype=int)
scored_frame = direction.merge(
    direction_scored.frame.drop_duplicates("variant_id")[
        ["variant_id"] + [f"score_{m}" for m in direction_ran]
    ],
    on="variant_id", how="left",
)
if len(scored_frame) != len(direction):
    raise ValueError(f"the join changed the row count: {len(direction)} -> {len(scored_frame)}")

# Which negatives each scorer actually saw, before any number is read.
print(f"{'scorer':<14}{'kind':>8}{'pos':>6}{'neg':>6}{'AUROC':>8}"
      f"{'  ' + str(int(CI * 100)) + '% CI':>16}{'p':>10}{'how':>22}")
print("-" * 92)
overall = []
for model in direction_ran:
    column = f"score_{model}"
    values = scored_frame[column].to_numpy(dtype=float)
    point, n_pos, n_neg = auroc(values, labels)
    low, high = auroc_ci(values, labels)
    p, floor, how = permutation_p(values, labels)
    kind = "matrix" if model in MATRIX_MODELS else "multi"
    interval = "-" if low != low else f"{low:.3f} to {high:.3f}"
    print(f"{model:<14}{kind:>8}{n_pos:>6}{n_neg:>6}{cell(point):>8}{interval:>16}"
          f"{('-' if p != p else f'{p:.4f}'):>10}{how:>22}")
    overall.append({"model": model, "kind": kind, "n_pos": n_pos, "n_neg": n_neg,
                    "auroc": point, "ci_low": low, "ci_high": high, "p": p,
                    "p_floor": floor})
overall = pd.DataFrame(overall).set_index("model")
print("-" * 92)
print("AUROC 0.5 is a coin. An interval that contains 0.5 is a scorer this panel")
print("cannot distinguish from one, which with 21 against 8 is most of them.")
print("'pos' and 'neg' are what that scorer could answer, not the panel size: a")
print("matrix scorer sees the singles only, so its row is a different test.")

# The weak-negative sensitivity check, as promised in Part 1.
keep = scored_frame["label"] != WEAK_NEGATIVE
print(f"\nthe same table with {WEAK_NEGATIVE} dropped "
      f"({int((~keep).sum())} variant, the one the source calls 'similar'):")
print(f"{'scorer':<14}{'pos':>6}{'neg':>6}{'AUROC':>8}{'delta':>9}")
print("-" * 43)
for model in direction_ran:
    column = f"score_{model}"
    point, n_pos, n_neg = auroc(scored_frame.loc[keep, column].to_numpy(dtype=float),
                                labels[keep.to_numpy()])
    before = overall.loc[model, "auroc"]
    delta = point - before
    print(f"{model:<14}{n_pos:>6}{n_neg:>6}{cell(point):>8}"
          f"{('-' if delta != delta else f'{delta:+.3f}'):>9}")
print("-" * 43)
print("A scorer whose AUROC moves a lot here was being carried by one weak label.")

In [ ]:
# --- the split that matters: singles against multi-mutation -----------------
# The matrix scorers cannot appear in the multi column at all - they return an
# empty cell for a candidate with several substitutions, by design, because a
# lookup would have to add single scores. So the two columns are two different
# tests on two different sets and the gap between them is not a degradation
# measurement; it is a statement about which scorers were asked which question.
singles = ~scored_frame["multi"].to_numpy()
multi = scored_frame["multi"].to_numpy()

print(f"singles: {int(labels[singles].sum())} improved, "
      f"{int((1 - labels[singles]).sum())} worsened")
print(f"multi:   {int(labels[multi].sum())} improved, "
      f"{int((1 - labels[multi]).sum())} worsened")
print()
head = (f"{'scorer':<14}{'singles':>9}{'pos/neg':>10}{'  CI':>16}"
        f"{'multi':>9}{'pos/neg':>10}{'  CI':>16}")
print(head)
print("-" * len(head))
split = []
for model in direction_ran:
    column = f"score_{model}"
    row = {"model": model}
    line = f"{model:<14}"
    for mask in (singles, multi):
        values = scored_frame.loc[mask, column].to_numpy(dtype=float)
        point, n_pos, n_neg = auroc(values, labels[mask])
        low, high = auroc_ci(values, labels[mask])
        interval = "-" if low != low else f"{low:.3f} to {high:.3f}"
        line += f"{cell(point):>9}{f'{n_pos}/{n_neg}':>10}{interval:>16}"
        suffix = "singles" if mask is singles else "multi"
        row[f"auroc_{suffix}"] = point
        row[f"n_{suffix}"] = f"{n_pos}/{n_neg}"
    print(line)
    split.append(row)
split = pd.DataFrame(split).set_index("model")
print("-" * len(head))
n_singles, n_multi = int(singles.sum()), int(multi.sum())
singles_floor = 1.0 / comb(n_singles, int((1 - labels[singles]).sum()))
print(f"The singles column is {int(labels[singles].sum())} against "
      f"{int((1 - labels[singles]).sum())}, so its best achievable p is")
print(f"1/C({n_singles},{int((1 - labels[singles]).sum())}) = {singles_floor:.4f} - "
      f"better than the old test's {old_floor:.4f}, and still not a number")
print(f"anybody should act on. The multi column is {int(labels[multi].sum())} "
      f"against {int((1 - labels[multi]).sum())}.")

# --- by format, to see whether the errors cluster there ---------------------
print(f"\n{'scorer':<14}", end="")
for fmt in sorted(scored_frame["format"].unique()):
    print(f"{fmt:>22}", end="")
print()
print("-" * (14 + 22 * scored_frame["format"].nunique()))
by_format = []
for model in direction_ran:
    column = f"score_{model}"
    print(f"{model:<14}", end="")
    row = {"model": model}
    for fmt in sorted(scored_frame["format"].unique()):
        mask = (scored_frame["format"] == fmt).to_numpy()
        point, n_pos, n_neg = auroc(scored_frame.loc[mask, column].to_numpy(dtype=float),
                                    labels[mask])
        print(f"{cell(point) + f' ({n_pos}/{n_neg})':>22}", end="")
        row[fmt] = point
    print()
    by_format.append(row)
by_format = pd.DataFrame(by_format).set_index("model")
print("-" * (14 + 22 * scored_frame["format"].nunique()))
print("'-' is a cell with too few of one label for an AUROC to exist, and Part 1")
print("said which cells those would be before anything was scored: scFv is")
print("negatives only and IgG is one positive against three. So the only format")
print("with a real AUROC is Fab, and the pooled number above is mostly it.")
print("That is a limit of the panel, not a finding about formats: this set cannot")
print("tell you whether a scorer transfers from Fab to IgG.")

for name, frame in (("overall", overall), ("by_class", split), ("by_format", by_format)):
    path = out_dir / "direction" / f"auroc_{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path)
    print(f"-> {path}")

## Part 3 - the Chinery binary

The Chinery 2026 block is the one place in the evidence set with enough labelled
candidates to make a real AUROC, and its label is **not** the Part 2 label. Read
this before the numbers, because the difference decides what the AUROC means.

720 CDR-H3 redesigns were submitted as IgG on a Carterra LSA. The negatives file
labels what came back:

| label | source value | wells | what it means |
|---|---|---|---|
| `positive` | `fitted_KD` | 156 | a clean 1:1 fit and a numeric KD |
| `negative_clean` | `bound_unfittable` | 97 | `I.C.` - binding was seen and could not be fit reliably |
| `negative_caveated` | `source_NA` | 434 | `N/A` - no fit at all |

So this is not "improved against the parent". It is **"bound well enough to be
fit"**, and `negative_clean` is the honest negative: those wells bound, visibly,
and too weakly or too heterogeneously to model - a weak binder, which is what a
scorer should be able to put below a measurable one. That is a real binary and
it is the one reported first.

`negative_caveated` is `N/A`, and **`N/A` is ambiguous in the source.** It is
consistent with a genuine non-binder, and equally consistent with a well that
failed to express, a sensor that did not regenerate, or a design that never
reached the chip. The evidence CSV is explicit that it does not convert `N/A`
into a demonstrated non-binder. Including those 434 adds 4.5x the negatives and
an unknown amount of label noise pointing in an unknown direction, so Part 3
reports **both** tables and neither as the answer.

**The base rate falls with substitution count, which is why the within-size
table exists.** Among the clean split it runs from about 70% positive at four
substitutions to 29% at ten. A pooled AUROC over that is partly measuring size,
exactly as the pooled Spearman in `03_chinery_size_calibration.ipynb` was, and a
within-size AUROC is read against the right null because inside a size bin the
base rate is fixed. The base rate per bin is printed beside every AUROC for the
same reason.

**And the whole block is selected on outcome.** The 156 are the designs that
produced a fit out of 720 submitted; the KD range is truncated and the
truncation is size-dependent. Every correlation in that notebook is attenuated
by it and so is every AUROC here. Each is a lower bound.

In [ ]:
# --- the Chinery block, one row per well ------------------------------------
chinery_rows = negatives_file[negatives_file["study"] == CHINERY_STUDY]
wanted = (CHINERY_POSITIVE, CHINERY_CLEAN, CHINERY_CAVEATED)
chinery_rows = chinery_rows[chinery_rows["label_class"].isin(wanted)]

records = []
for (well, label_class), group in chinery_rows.groupby(["variant", "label_class"]):
    mutations = [
        (str(r.chain), int(r.position), str(r.wt), str(r.mut)) for r in group.itertuples()
    ]
    declared = sorted(set(int(n) for n in group["n_mutations"]))
    if declared != [len(mutations)]:
        raise ValueError(f"{well}: {len(mutations)} rows but n_mutations says {declared}")
    records.append({
        "well": well, "label_class": label_class,
        "label": label_of(mutations), "n_mutations": len(mutations),
        "reason": sorted(set(group["label_reason"]))[0],
        "mutations": tuple(sorted(remap(mutations))),
    })
chinery = pd.DataFrame(records)
chinery["variant_id"] = [make_id(m) for m in chinery["mutations"]]
chinery["binary"] = (chinery["label_class"] == CHINERY_POSITIVE).astype(int)

counts = chinery.groupby("label_class")["well"].nunique().to_dict()
print(f"{CHINERY_STUDY}: {len(chinery)} wells over {chinery['variant_id'].nunique()} "
      f"distinct substitution sets")
for label_class in wanted:
    print(f"  {label_class:<20}{counts.get(label_class, 0):>5}  "
          f"({sorted(set(chinery.loc[chinery['label_class'] == label_class, 'reason']))})")
if counts != CHINERY_EXPECTED:
    raise ValueError(f"expected {CHINERY_EXPECTED}, got {counts}")

if chinery["variant_id"].duplicated().any():
    dupes = chinery.loc[chinery["variant_id"].duplicated(keep=False)].sort_values("variant_id")
    print(f"\n{dupes['variant_id'].nunique()} substitution sets appear in more than one "
          f"well; a scorer gives one number per set, so the duplicates would be scored "
          f"once and counted twice:")
    print(dupes[["well", "label_class", "n_mutations"]].to_string(index=False))
    raise ValueError(
        "two wells share a substitution set. Decide what the label of that set is "
        "before scoring it - if the two wells disagree, that is the assay's "
        "reproducibility and it belongs in the caveats, not averaged away here."
    )

# --- reconcile with the evidence CSV's own census ---------------------------
# The two files were curated separately and they do not have to agree. Print the
# gap rather than assert it away: the evidence CSV counts every submitted row,
# the negatives file has already dropped the wells it could not express as a
# substitution set (the length-changing ablang designs, and anything with no
# substitution list at all).
submitted = evidence[(evidence["study"] == CHINERY_STUDY)
                     & (evidence["format"] == "IgG")
                     & (evidence["record_type"] == "variant_outcome")]

# `raw_KD_M` holds the literal strings "N/A" and "I.C.", and "N/A" is in pandas'
# default list of missing-value markers - so in the frame read at the top of
# Part 1 every N/A well is already a NaN and `== "N/A"` is False everywhere.
# That is a silent miscount, not an error, so the column is re-read here with
# the default NA markers switched off and nothing else changed.
raw_status = pd.read_csv(
    EVIDENCE, low_memory=False, keep_default_na=False, na_values=[],
    usecols=["study", "format", "record_type", "raw_KD_M"],
)
raw_status = raw_status[(raw_status["study"] == CHINERY_STUDY)
                       & (raw_status["format"] == "IgG")
                       & (raw_status["record_type"] == "variant_outcome")]
if len(raw_status) != len(submitted):
    raise ValueError(f"the re-read selects {len(raw_status)} rows, not {len(submitted)}")
status = raw_status["raw_KD_M"].astype(str).str.strip()
census = {
    "numeric KD (a clean 1:1 fit)": int((submitted["metric"] == "KD").sum()),
    "I.C. (binding seen, not reliably fit)": int((status == "I.C.").sum()),
    "N/A (no fit)": int((status == "N/A").sum()),
    "blank": int((status == "").sum()),
}
if sum(census.values()) != len(submitted):
    raise ValueError(f"the census accounts for {sum(census.values())} of "
                     f"{len(submitted)} submitted designs")
print(f"\n{Path(EVIDENCE).name} census of the same block "
      f"({len(submitted)} submitted designs):")
pairs = ((CHINERY_POSITIVE, "numeric KD (a clean 1:1 fit)"),
         (CHINERY_CLEAN, "I.C. (binding seen, not reliably fit)"),
         (CHINERY_CAVEATED, "N/A (no fit)"))
print(f"{'label':<22}{'negatives file':>16}{'evidence file':>15}{'gap':>6}")
print("-" * 59)
for label_class, key in pairs:
    here, there = counts.get(label_class, 0), census[key]
    print(f"{label_class:<22}{here:>16}{there:>15}{there - here:>6}")
print(f"{'(not labelled)':<22}{'-':>16}{census['blank']:>15}{'':>6}")
print("-" * 59)
print(f"{'total':<22}{sum(counts.values()):>16}{len(submitted):>15}"
      f"{len(submitted) - sum(counts.values()):>6}")
print()
print("The gaps are the wells the negatives file could not express as a")
print("substitution set: the length-changing designs, whose CDR-H3 is nine")
print("residues instead of ten, and the ones with no substitution list at all. A")
print("variant_id here is a list of substitutions and cannot express an indel, so")
print("they are absent from both halves of the binary and do not bias it - they do")
print(f"mean this is {sum(counts.values())} of {len(submitted)} and not all of it.")

In [ ]:
# --- the base rate per substitution count, before any scorer is involved ----
def base_rates(frame, negative_classes):
    rows = []
    subset = frame[frame["label_class"].isin((CHINERY_POSITIVE,) + tuple(negative_classes))]
    for size in sorted(subset["n_mutations"].unique()):
        at_size = subset[subset["n_mutations"] == size]
        n_pos = int((at_size["binary"] == 1).sum())
        n_neg = int((at_size["binary"] == 0).sum())
        rows.append({"n_sub": int(size), "positive": n_pos, "negative": n_neg,
                     "total": n_pos + n_neg,
                     "base_rate": n_pos / float(n_pos + n_neg) if n_pos + n_neg else float("nan")})
    table = pd.DataFrame(rows).set_index("n_sub")
    return table


clean_rates = base_rates(chinery, (CHINERY_CLEAN,))
caveated_rates = base_rates(chinery, (CHINERY_CLEAN, CHINERY_CAVEATED))

print("base rate = share of wells at this size that returned a numeric KD")
print(f"{'n_sub':>6}{'pos':>6}{'neg':>6}{'total':>7}{'base rate':>11}"
      f"{'  |':>4}{'pos':>6}{'neg':>6}{'total':>7}{'base rate':>11}")
print(f"{'':>6}{'clean split (positive vs negative_clean)':^30}{'':>4}"
      f"{'with negative_caveated':^30}")
print("-" * 76)
for size in sorted(set(clean_rates.index) | set(caveated_rates.index)):
    line = f"{size:>6}"
    for table in (clean_rates, caveated_rates):
        if size in table.index:
            row = table.loc[size]
            line += (f"{int(row['positive']):>6}{int(row['negative']):>6}"
                     f"{int(row['total']):>7}{row['base_rate']:>10.1%}")
        else:
            line += f"{'-':>6}{'-':>6}{'-':>7}{'-':>11}"
        line += f"{'  |':>4}" if table is clean_rates else ""
    print(line)
print("-" * 76)
for name, table in (("clean", clean_rates), ("with caveated", caveated_rates)):
    pooled = table["positive"].sum() / float(table["total"].sum())
    trend = metrics.spearman(table.index.to_numpy(dtype=float),
                             table["base_rate"].to_numpy(dtype=float))
    print(f"{name:<15} pooled base rate {pooled:.1%}, "
          f"Spearman of base rate against size {trend:+.3f}")
print()
print("The base rate falls with size, so a pooled AUROC over the whole block is")
print("partly a size detector - the same confound the pooled Spearman had in")
print("03_chinery_size_calibration.ipynb. Inside a size bin the base rate is")
print("fixed, which is the only place 0.5 is the right null.")
print("Sizes 1 and 2 have 0 and 1 clean negatives, so CHINERY_SIZES starts at 3.")

In [ ]:
# --- score the Chinery block ------------------------------------------------
# 253 candidates for the clean split, 687 with the caveated negatives. For the
# per-variant scorers that is the whole cost of this notebook: FoldX is
# FOLDX_RUNS BuildModel runs plus two AnalyseComplex calls per candidate, so
# point DESIGN_PIPELINE_FOLDX_PDB at the trimmed, repaired Fv plus antigen
# domain the alanine scan used or it is days rather than hours.
classes = [CHINERY_POSITIVE, CHINERY_CLEAN]
if SCORE_CAVEATED:
    classes.append(CHINERY_CAVEATED)
to_score = chinery[chinery["label_class"].isin(classes)].reset_index(drop=True)
print(f"scoring {len(to_score)} candidates "
      f"({'clean split plus the caveated negatives' if SCORE_CAVEATED else 'the clean split only'})")
if not SCORE_CAVEATED:
    print("SCORE_CAVEATED is False, so the second table below is reported as not computed.")
if len(to_score) > MAX_VARIANTS:
    raise ValueError(
        f"{len(to_score)} candidates against MAX_VARIANTS={MAX_VARIANTS}. That cap is "
        f"what stops an uncapped per-variant scorer queueing a week of compute; raise "
        f"it deliberately, or set SCORE_CAVEATED = False."
    )

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="anarci is not installed")
    chinery_table = enumerate_variants(
        complex_, HEAVY_CHAIN, out_dir / "chinery" / "variants",
        only=to_score["variant_id"].tolist(), structure=pdb,
    )
print(f"{chinery_table.rows} rows, {chinery_table.candidates} candidates")
if chinery_table.candidates != len(to_score):
    raise ValueError(f"{chinery_table.candidates} candidates for {len(to_score)} wells")

print("\nscoring the Chinery block:")
chinery_scored, chinery_ran, chinery_skipped = score_set(
    chinery_table, ALL_MODELS, out_dir / "chinery" / "scores", "Chinery block"
)
chinery_panel = to_score.merge(
    chinery_scored.frame.drop_duplicates("variant_id")[
        ["variant_id"] + [f"score_{m}" for m in chinery_ran]
    ],
    on="variant_id", how="left",
)
if len(chinery_panel) != len(to_score):
    raise ValueError("the join changed the row count")
print(f"\nNote which scorers are absent here and why: every Chinery design carries")
print(f"at least one substitution and {int((chinery_panel['n_mutations'] > 1).sum())} of "
      f"{len(chinery_panel)} carry several, so the")
print("matrix scorers answer only the singles - 14 of 687 - and their rows below")
print("are thin by construction rather than by accident.")

In [ ]:
# --- the Chinery AUROC: overall, then within each substitution count --------
def chinery_auroc_table(negative_classes, title):
    subset = chinery_panel[
        chinery_panel["label_class"].isin((CHINERY_POSITIVE,) + tuple(negative_classes))
    ]
    y = subset["binary"].to_numpy(dtype=int)
    print(f"\n{title}")
    print(f"  {int((y == 1).sum())} positive, {int((y == 0).sum())} negative, "
          f"pooled base rate {y.mean():.1%}")
    head = (f"{'scorer':<14}{'pooled':>8}{'pos/neg':>10}{'  CI':>16}"
            + "".join(f"{f'n={s}':>9}" for s in CHINERY_SIZES))
    print(head)
    print("  " + "-" * (len(head) - 2))
    rows = []
    for model in chinery_ran:
        column = f"score_{model}"
        values = subset[column].to_numpy(dtype=float)
        point, n_pos, n_neg = auroc(values, y)
        low, high = auroc_ci(values, y)
        interval = "-" if low != low else f"{low:.3f} to {high:.3f}"
        line = f"{model:<14}{cell(point):>8}{f'{n_pos}/{n_neg}':>10}{interval:>16}"
        row = {"model": model, "auroc": point, "ci_low": low, "ci_high": high,
               "n_pos": n_pos, "n_neg": n_neg}
        for size in CHINERY_SIZES:
            mask = (subset["n_mutations"] == size).to_numpy()
            at_size, _, _ = auroc(subset.loc[mask, column].to_numpy(dtype=float), y[mask])
            line += f"{cell(at_size):>9}"
            row[f"auroc_n{size}"] = at_size
        print(line)
        rows.append(row)
    print("  " + "-" * (len(head) - 2))
    line = f"{'base rate':<14}{y.mean():>8.3f}{'':>10}{'':>16}"
    for size in CHINERY_SIZES:
        mask = (subset["n_mutations"] == size).to_numpy()
        line += f"{(y[mask].mean() if mask.any() else float('nan')):>9.3f}"
    print(line)
    print("  the base-rate row is not an AUROC - it is what share of that bin is")
    print("  positive, printed so each column is read against its own population.")
    print("  AUROC's null is 0.5 whatever the base rate; the base rate is what tells")
    print("  you whether a bin has enough of both labels to mean anything.")
    return pd.DataFrame(rows).set_index("model")


chinery_clean_auroc = chinery_auroc_table(
    (CHINERY_CLEAN,), "clean split: positive (fitted KD) against negative_clean (I.C.)"
)
if SCORE_CAVEATED:
    chinery_all_auroc = chinery_auroc_table(
        (CHINERY_CLEAN, CHINERY_CAVEATED),
        "with negative_caveated included: positive against I.C. plus N/A",
    )
    print("\nThe two tables answer different questions and the second one's negatives")
    print("are the ambiguous ones. 'N/A' in the source is consistent with a genuine")
    print("non-binder and equally consistent with a well that failed to express or a")
    print("sensor that did not regenerate; the evidence CSV refuses to call it a")
    print("demonstrated non-binder and so does this notebook. Where a scorer looks")
    print("better in the second table, the honest reading is that it separates")
    print("'something went wrong with this well' from 'this bound', which is not the")
    print("claim anybody wants to make.")
else:
    chinery_all_auroc = None
    print("\nSCORE_CAVEATED is False, so the caveated table was not computed. It is")
    print("not reported as 'no difference' - it was not run.")

for name, frame in (("clean", chinery_clean_auroc), ("with_caveated", chinery_all_auroc)):
    if frame is None:
        continue
    path = out_dir / "chinery" / f"auroc_{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path)
    print(f"-> {path}")
print()
print("And the caveat that attenuates both: the block is selected on outcome -")
print("156 fits out of 720 submitted - and the selection is size-dependent, so")
print("every AUROC here is a lower bound and the right-hand size columns are the")
print("survivors of a harsher filter than the left-hand ones.")

## Part 4 - manufactured negatives, as a floor check

Real negatives are scarce: Part 2 has eight of them and three share a
substitution. Confident negatives, on the other hand, are free - you can
construct a variant that no reasonable model should like and check whether the
scorer likes it. That is a weaker test than a measurement and it is a *different*
test, because it has no false-negative risk at all: nobody needs an SPR run to
know that putting a proline in the middle of a beta strand is bad.

**This part cannot tell you a scorer is good. It can tell you a scorer is
broken, and a scorer that fails here fails regardless of what Parts 2 and 3
said.** A scorer with AUROC 0.7 on eight negatives and a 60% failure rate on
manufactured ones has learned something other than binding.

Five classes, four of them deliberate breaks and one a control on the control:

* **Alanine at each of the ten FoldX hotspots.** `design_pipeline.hotspots`
  holds them: positions that lose at least 1.0 kcal/mol of binding on alanine,
  all within 3.8 A of HER2. Alanine removes the side chain past its first
  carbon, so these are the positions whose side chains carry the interaction.
  **FoldX is not scored on these** - the hotspot list *is* FoldX output, so
  asking FoldX to rank it is asking FoldX whether it agrees with itself.
  AntiFold, GearBind and the language models never saw it.
* **Physically implausible substitutions**, derived from the coordinates rather
  than from a list: a proline in the middle of a beta strand (proline has no
  amide hydrogen and the wrong phi, so it breaks the strand's hydrogen-bond
  ladder), a glycine replacing a buried bulky side chain (a cavity where a
  packed core used to be), and a charge reversal at a salt bridge (an attraction
  turned into a repulsion). None of these three comes from FoldX, so every
  scorer including FoldX is graded on them.
* **Non-interface surface substitutions, as neutral decoys.** Alanine at exposed
  positions outside the paratope. These are the control on the control: they
  **should score near zero**. A scorer that hands them large positive numbers is
  not distinguishing the interface from the bulk of the protein, and its
  enthusiasm for a real design is then enthusiasm about something else. FoldX is
  not scored on these either, for the same circularity reason as the hotspots.

**The sign convention is what makes a single threshold meaningful.**
`score_<model>` is higher-is-better-binding for every scorer in the registry,
and zero is the wild type *by construction* in every case: the matrix scorers
and `antifold_seq` report a log-likelihood ratio against wild type, FoldX and
GearBind report a negated ddG. So "scored above zero" means "the scorer calls
this deliberate break an improvement", and that is the failure rate reported
below. The harder criterion beside it is "scored above the median of the real
positives from Part 1", which is what a shortlist cut would actually act on.

And one more control that needs no manufactured variant at all, in the cell
after: `wt_argmax_rate`, which turns every interface position into its own
control.

In [ ]:
# --- the structure, read once for burial, secondary structure and contacts --
import biotite.structure as struc

# Theoretical maximum solvent-accessible side-chain area per residue, from
# Tien et al. 2013 (PLoS ONE 8:e80635, Table 1, theoretical values). Relative
# SASA is the residue's area over this, which is the usual definition of burial
# and is what BURIED_BELOW and SURFACE_ABOVE are thresholds on.
MAX_ASA = {
    "A": 129.0, "R": 274.0, "N": 195.0, "D": 193.0, "C": 167.0, "E": 223.0,
    "Q": 225.0, "G": 104.0, "H": 224.0, "I": 197.0, "L": 201.0, "K": 236.0,
    "M": 224.0, "F": 240.0, "P": 159.0, "S": 155.0, "T": 172.0, "W": 285.0,
    "Y": 263.0, "V": 174.0,
}

atoms = structures.load(pdb)
atoms = atoms[struc.filter_amino_acids(atoms)]
antibody = atoms[np.isin(atoms.chain_id, [HEAVY_CHAIN, LIGHT_CHAIN])]

# Burial and secondary structure are computed on the antibody alone, with HER2
# removed. That is deliberate: a buried core position should be buried by its own
# domain, not by the antigen, and a position buried only in the complex is an
# interface position - which the paratope set below already names.
#
# They are computed over the *whole* of both chains, because that is the real
# molecule and a residue's burial depends on what is packed against it. The
# variable-domain restriction is applied when the classes are built, not here:
# 1N8Z carries a whole light chain and a heavy Fd, so most of each chain is
# constant domain that this pipeline never designs in.
atom_sasa = struc.sasa(antibody, vdw_radii="Single")
residue_sasa = struc.apply_residue_wise(antibody, atom_sasa, np.nansum)
starts = struc.get_residue_starts(antibody)
residue_chain = antibody.chain_id[starts]

# Atom index -> residue index, which is the map the salt-bridge search needs:
# `starts` holds atom indices and every per-residue array below is indexed by
# residue. Conflating the two is the bug this exists to make impossible.
bounds = list(starts) + [antibody.array_length()]
residue_of_atom = np.empty(antibody.array_length(), dtype=int)
for residue, (begin, end) in enumerate(zip(bounds, bounds[1:])):
    residue_of_atom[begin:end] = residue

relative_sasa, sse_of, residue_at = {}, {}, {}
for chain in (HEAVY_CHAIN, LIGHT_CHAIN):
    picked = np.flatnonzero(residue_chain == chain)
    sequence = sequences[chain]
    if len(picked) != len(sequence):
        raise IndexMismatchError(
            f"chain {chain}: {len(picked)} residues in the SASA pass against "
            f"{len(sequence)} in chain_sequence. The two disagree about what a "
            f"residue is, so positions cannot be mapped between them."
        )
    for position, index in enumerate(picked, start=1):
        residue_at[int(index)] = (chain, position)
        reference = MAX_ASA.get(sequence[position - 1])
        relative_sasa[(chain, position)] = (
            float(residue_sasa[index]) / reference if reference else float("nan")
        )
    sub = antibody[antibody.chain_id == chain]
    sse = struc.annotate_sse(sub)
    if len(sse) != len(sequence):
        raise IndexMismatchError(
            f"chain {chain}: annotate_sse returned {len(sse)} assignments for "
            f"{len(sequence)} residues, so they cannot be lined up by order."
        )
    # biotite 2.0 returns PSeaElement integers, not letters, and to_symbols is
    # the documented way back to 'a' / 'b' / 'c'. Converted here rather than
    # compared against a hard-coded 2, which is the kind of thing that silently
    # becomes wrong when the enum gains a member.
    symbols = (struc.PSeaElement.to_symbols(sse)
               if hasattr(struc, "PSeaElement") else sse)
    for position, code in enumerate(symbols, start=1):
        sse_of[(chain, position)] = str(code)


def in_variable_domain(chain, position):
    return position <= VARIABLE_DOMAIN.get(chain, 0)


paratope = metrics.interface_residues(
    pdb, [HEAVY_CHAIN, LIGHT_CHAIN], [ANTIGEN_CHAIN], cutoff=PARATOPE_CUTOFF
)
outside = sorted(k for k in paratope if not in_variable_domain(*k))
print(f"paratope at {PARATOPE_CUTOFF} A: {len(paratope)} positions "
      f"({sum(1 for c, _ in paratope if c == HEAVY_CHAIN)} on {HEAVY_CHAIN}, "
      f"{sum(1 for c, _ in paratope if c == LIGHT_CHAIN)} on {LIGHT_CHAIN})")
if outside:
    print(f"  {len(outside)} of them are outside the variable domains: {outside}")
    print("  which would mean HER2 is contacting a constant domain in this")
    print("  structure - worth a look before trusting the decoy set, since the")
    print("  decoys are defined as 'not in the paratope'.")

print(f"\nchains {HEAVY_CHAIN} and {LIGHT_CHAIN}: {len(relative_sasa)} residues, "
      f"{sum(1 for k in relative_sasa if in_variable_domain(*k))} of them in the "
      f"variable domains (VH 1-{VARIABLE_DOMAIN[HEAVY_CHAIN]}, "
      f"VL 1-{VARIABLE_DOMAIN[LIGHT_CHAIN]})")
counts = pd.Series([v for k, v in sse_of.items() if in_variable_domain(*k)]).value_counts()
print(f"secondary structure over the variable domains (biotite P-SEA): "
      + ", ".join(f"{k}={v}" for k, v in counts.items())
      + "  (a = helix, b = strand, c = coil)")
buried = [k for k, v in relative_sasa.items()
          if v < BURIED_BELOW and in_variable_domain(*k)]
surface = [k for k, v in relative_sasa.items()
           if v > SURFACE_ABOVE and in_variable_domain(*k)]
print(f"relative SASA in the variable domains: {len(buried)} positions under "
      f"{BURIED_BELOW} (buried), {len(surface)} over {SURFACE_ABOVE} (exposed)")

# --- salt bridges, carboxyl oxygen to basic nitrogen ------------------------
ACID = {"ASP": ("OD1", "OD2"), "GLU": ("OE1", "OE2")}
BASE = {"LYS": ("NZ",), "ARG": ("NE", "NH1", "NH2"), "HIS": ("ND1", "NE2")}


def charged_atoms(table):
    names = np.array([str(n) for n in antibody.res_name])
    atom_names = np.array([str(n) for n in antibody.atom_name])
    keep = np.zeros(antibody.array_length(), dtype=bool)
    for residue, wanted in table.items():
        keep |= (names == residue) & np.isin(atom_names, list(wanted))
    return np.flatnonzero(keep)


acid_atoms, base_atoms = charged_atoms(ACID), charged_atoms(BASE)
bridges = set()
if len(acid_atoms) and len(base_atoms):
    gaps = np.linalg.norm(
        antibody.coord[acid_atoms][:, None, :] - antibody.coord[base_atoms][None, :, :],
        axis=-1,
    )
    for i, j in zip(*np.nonzero(gaps <= SALT_BRIDGE_CUTOFF)):
        acid_site = residue_at[int(residue_of_atom[acid_atoms[i]])]
        base_site = residue_at[int(residue_of_atom[base_atoms[j]])]
        if acid_site == base_site:
            continue
        bridges.add((acid_site, base_site))
in_fv = {pair for pair in bridges
         if in_variable_domain(*pair[0]) and in_variable_domain(*pair[1])}
print(f"\nsalt bridges within {SALT_BRIDGE_CUTOFF} A: {len(bridges)} acid-base "
      f"residue pairs over both whole chains, {len(in_fv)} of them with both "
      f"partners in the variable domains")
bridges = in_fv
print(f"  those {len(bridges)} pairs involve "
      f"{len({r for pair in bridges for r in pair})} residues")

In [ ]:
# --- build the five classes -------------------------------------------------
# One variant per row, with the class it belongs to, what it is expected to do
# and which scorers may be graded on it. A position that qualifies for two
# classes is kept in the first and the overlap is printed, so no manufactured
# variant is counted twice.
rng = np.random.default_rng(DECOY_SEED)
FOLDX_DERIVED = "the FoldX alanine scan"


def capped(items, limit=None):
    '''A seeded sample, so re-running the notebook builds the same set.'''
    limit = MAX_PER_CLASS if limit is None else limit
    items = sorted(items)
    if len(items) <= limit:
        return items
    picked = rng.choice(len(items), size=limit, replace=False)
    return [items[i] for i in sorted(picked)]


proposals = []


def propose(chain, position, mut, group, expect, excludes, why):
    wt = sequences[chain][position - 1]
    if wt == mut:
        return
    proposals.append({
        "group": group, "expect": expect, "excludes": excludes,
        "chain": chain, "position": position, "wt": wt, "mut": mut,
        "label": f"{chain}:{wt}{position}{mut}", "why": why,
        "rel_sasa": relative_sasa.get((chain, position), float("nan")),
        "sse": sse_of.get((chain, position), "?"),
        "in_paratope": (chain, position) in paratope,
    })


# 1. Alanine at the ten FoldX hotspots.
hotspot_rows = [h for h in hotspots.HOTSPOTS
                if h.status == "hotspot" and h.ddg_bind is not None
                and h.ddg_bind >= HOTSPOT_DDG]
print(f"{len(hotspot_rows)} hotspots at or above {HOTSPOT_DDG} kcal/mol on alanine:")
for h in hotspot_rows:
    print(f"  {h.crystal_id:<8}{h.domain:<5}ddg_bind {h.ddg_bind:>5.2f} "
          f"({h.runs} runs)")
    if sequences[h.chain][h.position - 1] != h.wt:
        raise IndexMismatchError(
            f"hotspots says {h.crystal_id} but 1N8Z has "
            f"{sequences[h.chain][h.position - 1]} at {h.chain}:{h.position}"
        )
    propose(h.chain, h.position, "A", "hotspot_alanine", "break", FOLDX_DERIVED,
            f"alanine costs {h.ddg_bind:.2f} kcal/mol of binding (FoldX scan)")

# 2a. Proline in the middle of a beta strand: both neighbours also strand, so
# this is not an edge position where a proline is tolerated.
mid_strand = [
    (chain, position)
    for (chain, position), code in sorted(sse_of.items())
    if code == "b"
    and sse_of.get((chain, position - 1)) == "b"
    and sse_of.get((chain, position + 1)) == "b"
    and sequences[chain][position - 1] not in "PG"
    and in_variable_domain(chain, position)
]
for chain, position in capped(mid_strand):
    propose(chain, position, "P", "proline_in_strand", "break", None,
            "mid-strand: both neighbours are strand, so the hydrogen-bond ladder breaks")

# 2b. Glycine replacing a buried bulky side chain.
buried_bulky = [
    (chain, position)
    for (chain, position), value in sorted(relative_sasa.items())
    if value < BURIED_BELOW
    and sequences[chain][position - 1] in BULKY
    and in_variable_domain(chain, position)
]
for chain, position in capped(buried_bulky):
    propose(chain, position, "G", "glycine_for_buried_bulky", "break", None,
            f"relative SASA {relative_sasa[(chain, position)]:.3f}, a packed core position")

# 2c. Charge reversal at a salt bridge: acid to lysine, base to aspartate.
reversals = []
for (acid, base) in sorted(bridges):
    reversals.append((acid, "K", f"salt bridge with {base[0]}:{base[1]}"))
    reversals.append((base, "D", f"salt bridge with {acid[0]}:{acid[1]}"))
seen = set()
unique_reversals = []
for (chain, position), mut, why in reversals:
    if (chain, position) in seen:
        continue
    seen.add((chain, position))
    unique_reversals.append(((chain, position), mut, why))
for (chain, position), mut, why in capped(
    [(k, m, w) for k, m, w in unique_reversals]
):
    propose(chain, position, mut, "charge_reversal", "break", None,
            f"charge reversal, {why}")

# 3. Neutral decoys: alanine at exposed positions outside the paratope.
decoy_sites = [
    (chain, position)
    for (chain, position), value in sorted(relative_sasa.items())
    if value > SURFACE_ABOVE
    and (chain, position) not in paratope
    and sequences[chain][position - 1] != "A"
    and in_variable_domain(chain, position)
]
for chain, position in capped(decoy_sites, DECOY_POSITIONS):
    propose(chain, position, "A", "surface_decoy", "neutral", FOLDX_DERIVED,
            f"exposed (relative SASA {relative_sasa[(chain, position)]:.3f}) and "
            f"outside the paratope")

manufactured = pd.DataFrame(proposals)
overlaps = manufactured[manufactured.duplicated(["chain", "position", "mut"], keep=False)]
if len(overlaps):
    print(f"\n{overlaps['label'].nunique()} substitution(s) proposed by more than one "
          f"class; kept in the first and dropped from the rest:")
    print(overlaps[["label", "group", "why"]].to_string(index=False))
manufactured = manufactured.drop_duplicates(["chain", "position", "mut"], keep="first")
manufactured["variant_id"] = [
    make_id([(r.chain, r.position, r.wt, r.mut)]) for r in manufactured.itertuples()
]

# None of these may coincide with a measured variant: a manufactured negative
# that is actually in the panel is a measurement, and the two must not be mixed.
measured = set(direction["variant_id"]) | set(chinery["variant_id"])
clash = manufactured[manufactured["variant_id"].isin(measured)]
if len(clash):
    print(f"\n{len(clash)} manufactured variant(s) are also measured somewhere in the "
          f"benchmark and are dropped from this part:")
    print(clash[["label", "group", "why"]].to_string(index=False))
    manufactured = manufactured[~manufactured["variant_id"].isin(measured)]
manufactured = manufactured.reset_index(drop=True)

print(f"\n{len(manufactured)} manufactured variants:")
summary = (manufactured.groupby(["group", "expect"])
           .agg(n=("label", "size"),
                in_paratope=("in_paratope", "sum"),
                median_rel_sasa=("rel_sasa", "median"))
           .reset_index())
print(summary.to_string(index=False))
print(f"\n'excludes' is which scorer may not be graded on a class because the class")
print(f"came out of that scorer:")
for group, excludes in manufactured.groupby("group")["excludes"].first().items():
    print(f"  {group:<28}{excludes if isinstance(excludes, str) else '(nothing)'}")
print(manufactured[["label", "group", "expect", "sse", "rel_sasa", "in_paratope", "why"]]
      .to_string(index=False))

In [ ]:
# --- score the manufactured set ---------------------------------------------
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="anarci is not installed")
    manufactured_table = enumerate_variants(
        complex_, HEAVY_CHAIN, out_dir / "manufactured" / "variants",
        only=manufactured["variant_id"].tolist(), structure=pdb,
    )
print(f"{manufactured_table.rows} rows, {manufactured_table.candidates} candidates")

print("\nscoring the manufactured set:")
manufactured_scored, manufactured_ran, manufactured_skipped = score_set(
    manufactured_table, ALL_MODELS, out_dir / "manufactured" / "scores", "manufactured set"
)
manufactured_panel = manufactured.merge(
    manufactured_scored.frame.drop_duplicates("variant_id")[
        ["variant_id"] + [f"score_{m}" for m in manufactured_ran]
    ],
    on="variant_id", how="left",
)

# Every manufactured variant is a single substitution, so every scorer can
# answer every one of them - which is the point: this is the one part of the
# notebook where the matrix scorers are tested on their full set.
if (manufactured_panel["position"] <= 0).any():
    raise ValueError("a manufactured variant has a non-positive position")
print(f"\nall {len(manufactured_panel)} manufactured variants are single substitutions, "
      f"so no scorer is excused by the lookup rule here.")

# The reference the harder criterion is measured against: the median score of
# the real positives from Part 1, per scorer.
positive_median = {}
for model in manufactured_ran:
    column = f"score_{model}"
    if column not in scored_frame.columns:
        positive_median[model] = float("nan")
        continue
    values = scored_frame.loc[scored_frame["direction"] == 1, column].to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    positive_median[model] = float(np.median(values)) if len(values) else float("nan")
print(f"\nthe median score of Part 1's measured improvers, per scorer, which is the")
print(f"bar the second criterion uses:")
for model, value in positive_median.items():
    n = int(np.isfinite(scored_frame.get(f'score_{model}', pd.Series(dtype=float))
                        .where(scored_frame['direction'] == 1)).sum())
    print(f"  {model:<14}{('-' if value != value else f'{value:+.3f}'):>10}  "
          f"over {n} scored improvers")

In [ ]:
# --- the failure rates ------------------------------------------------------
# Two criteria per class. `calls improving` is the share scored above
# FAILURE_ABOVE, which is the wild type by construction. `beats positives` is
# the share scored above the median of Part 1's measured improvers, which is
# what a shortlist cut would act on. The second is the one that matters and the
# first is the one that is unambiguous.
breaks = manufactured_panel[manufactured_panel["expect"] == "break"]
decoys = manufactured_panel[manufactured_panel["expect"] == "neutral"]

rows = []
groups = list(manufactured_panel.groupby("group", sort=True).groups)
head = f"{'scorer':<14}{'class':<28}{'n':>4}{'calls improving':>17}{'beats positives':>17}{'median score':>14}"
print(head)
print("-" * len(head))
for model in manufactured_ran:
    column = f"score_{model}"
    for group in groups:
        subset = manufactured_panel[manufactured_panel["group"] == group]
        excludes = subset["excludes"].iloc[0]
        if isinstance(excludes, str) and model == "foldx":
            print(f"{model:<14}{group:<28}{'-':>4}{'not scored':>17}"
                  f"{'(circular)':>17}{'-':>14}")
            continue
        values = subset[column].to_numpy(dtype=float)
        usable = values[np.isfinite(values)]
        if not len(usable):
            print(f"{model:<14}{group:<28}{0:>4}{'-':>17}{'-':>17}{'-':>14}")
            continue
        improving = float((usable > FAILURE_ABOVE).mean())
        bar = positive_median[model]
        beats = float((usable > bar).mean()) if bar == bar else float("nan")
        print(f"{model:<14}{group:<28}{len(usable):>4}{improving:>16.0%}"
              f"{('-' if beats != beats else f'{beats:.0%}'):>17}"
              f"{np.median(usable):>+14.3f}")
        rows.append({"model": model, "group": group, "expect": subset["expect"].iloc[0],
                     "n": len(usable), "calls_improving": improving,
                     "beats_positives": beats, "median": float(np.median(usable))})
    print("-" * len(head))
failures = pd.DataFrame(rows)

# The headline per scorer: the failure rate over the deliberate breaks only,
# with the decoys reported separately because their expectation is different.
print(f"\n{'scorer':<14}{'breaks n':>10}{'failure rate':>14}{'beats positives':>17}"
      f"{'decoys n':>10}{'decoy median':>14}{'decoys > 0':>12}")
print("-" * 91)
manufactured_summary = []
for model in manufactured_ran:
    column = f"score_{model}"
    allowed = breaks[~((breaks["excludes"].notna()) & (model == "foldx"))]
    values = allowed[column].to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    bar = positive_median[model]
    rate = float((values > FAILURE_ABOVE).mean()) if len(values) else float("nan")
    beats = float((values > bar).mean()) if len(values) and bar == bar else float("nan")
    decoy_allowed = decoys[~((decoys["excludes"].notna()) & (model == "foldx"))]
    decoy_values = decoy_allowed[column].to_numpy(dtype=float)
    decoy_values = decoy_values[np.isfinite(decoy_values)]
    decoy_median = float(np.median(decoy_values)) if len(decoy_values) else float("nan")
    decoy_above = float((decoy_values > FAILURE_ABOVE).mean()) if len(decoy_values) else float("nan")
    print(f"{model:<14}{len(values):>10}"
          f"{('-' if rate != rate else f'{rate:.0%}'):>14}"
          f"{('-' if beats != beats else f'{beats:.0%}'):>17}"
          f"{len(decoy_values):>10}"
          f"{('-' if decoy_median != decoy_median else f'{decoy_median:+.3f}'):>14}"
          f"{('-' if decoy_above != decoy_above else f'{decoy_above:.0%}'):>12}")
    manufactured_summary.append({
        "model": model, "breaks_n": len(values), "failure_rate": rate,
        "breaks_beat_positives": beats, "decoys_n": len(decoy_values),
        "decoy_median": decoy_median, "decoys_above_zero": decoy_above,
    })
manufactured_summary = pd.DataFrame(manufactured_summary).set_index("model")
print("-" * 91)
print(f"'failure rate' is the share of deliberate breaks the scorer puts above the")
print(f"wild type. Read it against diagnostics.PLAUSIBLE_BENEFICIAL = "
      f"{diagnostics.PLAUSIBLE_BENEFICIAL:.0%}, which is")
print("this repo's loose upper bound on what share of substitutions to a matured")
print("antibody can plausibly be improvements - and these are not plausible ones.")
print()
print("'decoy median' should be near zero and 'decoys > 0' near half. A scorer with")
print("a large positive decoy median is enthusiastic about substitutions outside the")
print("paratope, which means its enthusiasm for a real design is about something")
print("other than the interface.")

path = out_dir / "manufactured" / "failures.csv"
path.parent.mkdir(parents=True, exist_ok=True)
failures.to_csv(path, index=False)
print(f"\n-> {path}")

### `wt_argmax_rate` - every interface position as its own control

The cheapest control in the set, and the only one that needs no negative at
all. Trastuzumab is a matured, clinically optimised antibody. At a paratope
position, a model that understands this molecule should mostly prefer **the
residue that is already there** over the other nineteen.

`design_pipeline.metrics.diagnostics.wt_argmax_rate` measures exactly that, and
the docstring is blunt about what a low rate means: it does not weaken the run,
it invalidates it - the model is not reading this antibody, and every number in
the column is then about something else. The threshold to read it against is
`EXPECTED_WT_ARGMAX`, 0.6.

Two mechanical points:

* **It needs the full alphabet at a position**, so this is a separate matrix run
  over the paratope only: nineteen substitutions at each interface position.
  `enumerate_variants` skips wild-type-to-wild-type because it is not a variant,
  so the wild type's own row is added here with a score of **exactly zero** -
  which is not a convenience, it is the definition. Every matrix scorer in this
  repo reports `M[position, mut] - M[position, wt]`, so the wild type's score
  against itself is zero by construction and not by assumption.
* **Matrix scorers only.** On 1N8Z at 5 A the paratope is 21 positions, so the
  full alphabet is 399 candidates - one forward pass for `antifold`, one pass
  per position for the language models, and nothing for the rest.
  `antifold_seq`, `foldx` and `gearbind` are one pass *per candidate*, so the
  same table would be 399 runs of a thing that takes seconds to minutes each.
  The question is also not quite theirs: they condition on the whole mutant, so
  "the best residue at this position" is not a quantity they compute, it is one
  you would have to assemble from 399 separate answers.

In [ ]:
# --- the paratope, the full alphabet, one matrix run ------------------------
interface_ids = []
for chain, position in sorted(paratope):
    wt = sequences[chain][position - 1]
    if wt == "X":
        continue
    for mut in AMINO_ACIDS:
        if mut == wt:
            continue
        interface_ids.append(make_id([(chain, position, wt, mut)]))
print(f"{len(sorted(paratope))} paratope positions x 19 substitutions = "
      f"{len(interface_ids)} candidates")

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="anarci is not installed")
    interface_table = enumerate_variants(
        complex_, HEAVY_CHAIN, out_dir / "manufactured" / "interface_variants",
        only=interface_ids, structure=pdb,
    )
print(f"{interface_table.rows} rows, {interface_table.candidates} candidates")

print("\nscoring the paratope alphabet (matrix scorers only):")
interface_scored, interface_ran, interface_skipped = score_set(
    interface_table, MATRIX_MODELS, out_dir / "manufactured" / "interface_scores",
    "paratope alphabet"
)

# The wild-type rows, at the score zero means by construction. Not a fill and
# not an imputation: every matrix scorer here reports M[position, mut] minus
# M[position, wt], so the wild type against itself is exactly 0.0.
wt_rows = pd.DataFrame([
    {"variant_id": make_id([(chain, position, sequences[chain][position - 1],
                             sequences[chain][position - 1])]),
     "chain": chain, "position": position,
     "wt": sequences[chain][position - 1], "mut": sequences[chain][position - 1]}
    for chain, position in sorted(paratope)
    if sequences[chain][position - 1] != "X"
])
for model in interface_ran:
    wt_rows[f"score_{model}"] = 0.0

columns = ["variant_id", "chain", "position", "wt", "mut"] + \
          [f"score_{m}" for m in interface_ran]
with_wt = pd.concat(
    [interface_scored.frame.reindex(columns=[c for c in columns
                                             if c in interface_scored.frame.columns]),
     wt_rows.reindex(columns=columns)],
    ignore_index=True,
)
print(f"\n{len(with_wt)} rows: {len(interface_scored.frame)} scored substitutions plus "
      f"{len(wt_rows)} wild-type rows at score 0.0 (the definition, not a fill)")

print(f"\n{'scorer':<14}{'positions':>11}{'scored':>8}{'skipped':>9}{'wt argmax rate':>16}"
      f"{'  verdict':>12}")
print("-" * 72)
wt_argmax = []
for model in interface_ran:
    result = diagnostics.wt_argmax_rate(with_wt, f"score_{model}")
    rate = result["rate"]
    verdict = ("-" if rate != rate
               else "ok" if rate >= diagnostics.EXPECTED_WT_ARGMAX else "INVALID")
    print(f"{model:<14}{result['positions']:>11}{result['scored']:>8}"
          f"{result['skipped']:>9}{('-' if rate != rate else f'{rate:.1%}'):>16}"
          f"{verdict:>12}")
    wt_argmax.append({"model": model, "positions": result["positions"],
                      "scored": result["scored"], "wt_argmax_rate": rate})
wt_argmax = pd.DataFrame(wt_argmax).set_index("model")
print("-" * 72)
print(f"read against diagnostics.EXPECTED_WT_ARGMAX = "
      f"{diagnostics.EXPECTED_WT_ARGMAX:.0%}. Materially below it and the column is")
print("not about this antibody, whatever its AUROC in Parts 2 and 3 came out at.")
print()
print("One honest complication: at a paratope position the residue that is there")
print("was selected for binding HER2, and AntiFold with no antigen in its input")
print("is conditioned on the Fab alone - so it is being asked to prefer a residue")
print("chosen for a reason it cannot see. A rate below the threshold is therefore")
print("weaker evidence of a bug for the structure-based scorers than for the")
print("sequence-based ones, which see the matured sequence itself.")

## Part 5 - the verdict table

One row per scorer. Four columns of evidence and one verdict, and the verdict is
a conjunction of stated criteria rather than a judgement:

| criterion | threshold | why that one |
|---|---|---|
| direction AUROC | the lower end of the bootstrap CI above 0.5 | the point estimate on 21 against 8 is not evidence; the interval is the claim |
| Chinery AUROC | the lower end of the bootstrap CI above 0.5 | the same rule on the panel large enough for the interval to be narrow |
| manufactured failure rate | at or below `PLAUSIBLE_BENEFICIAL` (25%) | this repo's existing bound on what share of substitutions can plausibly be improvements, and these are not plausible ones |
| `wt_argmax_rate` | at or above `EXPECTED_WT_ARGMAX` (60%), matrix scorers only | below it, the column is not about this antibody |

A scorer passes only if every criterion that applies to it passes. A criterion
that could not be measured - no route on this box, too few of one label - is
reported as not measured and the verdict is `unproven`, which is not `fail`:
nothing was shown either way.

**And the multiple-comparisons point, since Part 5 is where a winner gets
picked.** Several scorers are tested against the same small panels. With `k`
scorers the chance that at least one clears 0.5 by luck grows with `k`, and no
correction is applied below because applying one to 21-against-8 would be
decorating a number that cannot carry it. Instead the count is printed beside
the result: **whichever scorer wins, it won against this many others on this
much data**, and that is the sentence that should travel with it.

In [ ]:
# --- the verdict ------------------------------------------------------------
rows = []
for model in ALL_MODELS:
    row = {"model": model,
           "kind": "matrix" if model in MATRIX_MODELS else "multi"}
    if model in overall.index:
        row["direction_auroc"] = overall.loc[model, "auroc"]
        row["direction_ci_low"] = overall.loc[model, "ci_low"]
        row["direction_ci_high"] = overall.loc[model, "ci_high"]
        row["direction_p"] = overall.loc[model, "p"]
    else:
        row.update({"direction_auroc": float("nan"), "direction_ci_low": float("nan"),
                    "direction_ci_high": float("nan"), "direction_p": float("nan")})
    if model in chinery_clean_auroc.index:
        row["chinery_auroc"] = chinery_clean_auroc.loc[model, "auroc"]
        row["chinery_ci_low"] = chinery_clean_auroc.loc[model, "ci_low"]
        row["chinery_ci_high"] = chinery_clean_auroc.loc[model, "ci_high"]
    else:
        row.update({"chinery_auroc": float("nan"), "chinery_ci_low": float("nan"),
                    "chinery_ci_high": float("nan")})
    if model in manufactured_summary.index:
        row["failure_rate"] = manufactured_summary.loc[model, "failure_rate"]
        row["decoy_median"] = manufactured_summary.loc[model, "decoy_median"]
    else:
        row.update({"failure_rate": float("nan"), "decoy_median": float("nan")})
    row["wt_argmax"] = (wt_argmax.loc[model, "wt_argmax_rate"]
                        if model in wt_argmax.index else float("nan"))
    rows.append(row)
verdict = pd.DataFrame(rows).set_index("model")


def judge(row):
    '''(verdict, reasons). Every criterion that applies has to pass.'''
    failed, unmeasured = [], []
    for name, value, test in (
        ("direction", row["direction_ci_low"], lambda v: v > 0.5),
        ("chinery", row["chinery_ci_low"], lambda v: v > 0.5),
        ("manufactured", row["failure_rate"],
         lambda v: v <= diagnostics.PLAUSIBLE_BENEFICIAL),
    ):
        if value != value:
            unmeasured.append(name)
        elif not test(value):
            failed.append(name)
    if row["kind"] == "matrix":
        if row["wt_argmax"] != row["wt_argmax"]:
            unmeasured.append("wt_argmax")
        elif row["wt_argmax"] < diagnostics.EXPECTED_WT_ARGMAX:
            failed.append("wt_argmax")
    if failed:
        return "FAIL", "fails " + ", ".join(failed)
    if unmeasured:
        return "unproven", "not measured: " + ", ".join(unmeasured)
    return "pass", "every criterion met"


head = (f"{'scorer':<14}{'kind':>7}{'direction AUROC (CI)':>26}"
        f"{'Chinery AUROC (CI)':>24}{'manuf. fail':>13}{'wt argmax':>11}"
        f"{'  verdict':<11}")
print(head)
print("-" * (len(head) + 26))
for model, row in verdict.iterrows():
    call, reason = judge(row)
    verdict.loc[model, "verdict"] = call
    verdict.loc[model, "reason"] = reason
    direction_cell = (
        "not scored" if row["direction_auroc"] != row["direction_auroc"]
        else f"{row['direction_auroc']:.3f} ({row['direction_ci_low']:.2f}-{row['direction_ci_high']:.2f})"
    )
    chinery_cell = (
        "not scored" if row["chinery_auroc"] != row["chinery_auroc"]
        else f"{row['chinery_auroc']:.3f} ({row['chinery_ci_low']:.2f}-{row['chinery_ci_high']:.2f})"
    )
    print(f"{model:<14}{row['kind']:>7}{direction_cell:>26}{chinery_cell:>24}"
          f"{('-' if row['failure_rate'] != row['failure_rate'] else f'{row.failure_rate:.0%}'):>13}"
          f"{('-' if row['wt_argmax'] != row['wt_argmax'] else f'{row.wt_argmax:.0%}'):>11}"
          f"  {call:<9}{reason}")
print("-" * (len(head) + 26))

tested = int(verdict[["direction_auroc", "chinery_auroc"]].notna().any(axis=1).sum())
passed = verdict.index[verdict["verdict"] == "pass"].tolist()
unproven = verdict.index[verdict["verdict"] == "unproven"].tolist()
print(f"\n{tested} scorer(s) were measured on at least one panel here, of "
      f"{len(ALL_MODELS)} named in the parameters.")
if passed:
    print(f"pass: {', '.join(passed)} - and they passed against {tested - 1} others on "
          f"{len(direction)} measured variants and {int(chinery_clean_auroc['n_pos'].max() + chinery_clean_auroc['n_neg'].max()) if len(chinery_clean_auroc) else 0} Chinery wells.")
    print("That sentence travels with the result. Testing several scorers against one")
    print("small panel and reporting the best is how a null result becomes a claim.")
else:
    print("pass: none. No scorer in this repo is shown by these panels to be usable")
    print("for ranking, which is a statement about the evidence as much as about the")
    print("scorers: a 21-against-8 panel cannot establish an AUROC whose interval")
    print("excludes 0.5 unless the scorer is nearly perfect.")
if unproven:
    print(f"unproven: {', '.join(unproven)} - something could not be measured here, "
          f"which is not the same as a failure.")
for model, reason in verdict.loc[verdict["verdict"] == "unproven", "reason"].items():
    print(f"  {model:<14}{reason}")

path = out_dir / "verdict.csv"
verdict.to_csv(path)
print(f"\n-> {path}")

## What this notebook licenses, and what it does not

Fill the first paragraph in from the run. The rest is true before it runs,
because it is about the panels and not about the results.

**The claim the verdict table supports** is a negative one at the level of a
filter: a scorer whose direction interval contains 0.5, or whose manufactured
failure rate is above a quarter, should not be cutting a shortlist. That is
actionable without any positive claim about any scorer.

**The claim it does not support** is a ranking between scorers. Two scorers
whose intervals overlap are not ordered by this panel, and almost all of them
overlap.

### The caveats, restated so they travel with the numbers

* **`L:R66G` is a weak negative.** The source paper describes the affinity as
  *similar* and reports **4.8 +/- 0.7 nM against 2.0 +/- 0.4 nM with no
  significance test.** It is one of eight negatives and one of three singles, so
  it carries real weight in Part 2; the sensitivity table there reports every
  AUROC again with it dropped, and that delta is the honest measure of how much
  of the result rests on it.

* **The Chinery block is selected on outcome, so correlations there are
  attenuated.** 156 clean fits out of 720 submitted. Both the weakest and the
  unmeasurable candidates are gone, the dynamic range is truncated, and the
  truncation is size-dependent - a bigger design is likelier to fail to fit at
  all, so the large-size bins are the survivors of a harsher filter. Every AUROC
  in Part 3 is a lower bound, and the within-size columns on the right-hand side
  describe a different population from the ones on the left.

* **The `N/A` label is ambiguous in the source** and the evidence CSV refuses to
  call it a demonstrated non-binder. Part 3 reports the table with and without
  those 434 wells and neither as the answer. A scorer that looks better with
  them in may be separating "this well went wrong" from "this bound".

* **Several scorers were tested against small panels.** Part 5 prints how many,
  beside whichever wins. No multiplicity correction is applied, because
  correcting a 21-against-8 panel would be dressing up a number that cannot
  carry the weight; naming the count is the honest alternative.

* **Direction is pooled, magnitude is not.** Three formats - Fab, IgG, scFv -
  and three assays. The sign is comparable and nothing else is, which is why
  there is no KD, no fold change and no ddG anywhere in Parts 2 to 5.

* **The manufactured negatives are not a measurement.** They can show a scorer
  is broken. They cannot show one works, and a scorer that passes Part 4 has
  only been shown not to be obviously wrong. FoldX is excluded from the two
  alanine classes because the hotspot list is its own output; it is graded on the
  other three, which came from the coordinates.

### What would actually settle it

* **More measured negatives, from one format and one assay.** Eight negatives
  across three formats is the binding constraint on everything above, and the
  interval does not narrow until that number grows. Thirty Fab losses on one
  instrument against one batch-matched parent would do more than any modelling
  change in this repo.
* **A design set not filtered on whether it bound.** Even a coarse three-bin
  readout over all 720 Chinery designs would be worth more here than 156 clean
  KDs, because it is the 564 missing ones that carry the size dependence.
* **A positive set with repeats of one substitution in several contexts.**
  `H:D102W` is the best single in the panel, a component of an 8-fold
  improvement, and a component of a loss. One substitution measured in five
  backgrounds would turn that anecdote into a number, and that number is the
  ceiling on every additive scorer in the registry.